# Follow-ups to the R1 pilot

**Idea 1:** keep an internal factor fixed; follow its observable meaning.
**Idea 2:** keep an observable rule fixed; follow its internal representation.

Sections below expose each scientific calculation; the run-mode section at the
end wires them to the cached R1 objects. The legacy diagnosis is retrospective;
redesigned choices use reference patients only and are frozen before `evaluate`.
The original R1 artifacts stay read-only. Outcomes through 2014 are eligible for
primary analysis; 2015 remains available for representation diagnostics.
References 2007–2013 are primary; 2014/2015 are an added follow-up block.

| Mode | Purpose | Current availability |
|---|---|---|
| v0 | Tiny CPU invariants | Synthetic checks of every section |
| v1 | Bounded real-data GPU smoke | ≤128 rows, 5-epoch fit, replay/removal/domain |
| develop | Reference-only development and retrospective diagnosis | Writes `freeze.json` |
| evaluate | Frozen final suite | Needs `PILOT_FOLLOW_UPS_FREEZE` |
| report | Tables/figures of saved develop/evaluate outputs, all variants | Needs `PILOT_FOLLOW_UPS_RUNS`; no data or model work |

Run from the project root with `RPI_OUT_DIR` outside the checkout. Optional data
locations come from `PILOT_FOLLOW_UPS_POPULATION` and `PILOT_FOLLOW_UPS_R1`.
`PILOT_FOLLOW_UPS_VARIANT` picks `r1` (default), `baseline`, `last_3` or
`all_history`; non-R1 variants read legacy builds under `PILOT_FOLLOW_UPS_BUILDS`.
Each reference reads the build whose predictor context it shares: 2007 is R1 for
every variant, baseline 2007–2013 is R1, and `last_3` 2008/2009 equals `all_history`.
Default device is `auto`; v0 explicitly uses CPU. Device selection happens once.
Initial GPU batches are 16 inference, 4 gradients, 256 encoding, 128 training;
AMP and TF32 remain off. v0 loads no data.


In [ ]:
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import io
import json
import logging
import os
from pathlib import Path
import platform
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import time

import numpy as np
import torch

log = logging.getLogger("pilot_follow_ups")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
ROOT = Path.cwd()
MODE = os.environ.get("PILOT_FOLLOW_UPS_MODE", "v0")
if MODE not in {"v0", "v1", "develop", "evaluate", "report"}:
    raise ValueError(f"Unknown notebook mode: {MODE}")
CFG = {
    "mode": MODE, "device": os.environ.get("PILOT_FOLLOW_UPS_DEVICE", "auto"),
    "seed": 42, "split_seed": 42, "deterministic": False,
    "inference_batch": 16, "gradient_batch": 4, "encoding_batch": 256,
    "training_batch": 128, "cpu_workers": 4, "amp": "off", "tf32": False,
    "min_free_disk_bytes": 15 * 1024**3,
    "probability_rtol": 1e-4, "probability_atol": 1e-5,
    "gradient_rtol": 1e-3, "gradient_atol": 1e-5,
    "primary_outcome_last_year": 2014, "prospective_enabled": False, "prospective_signal": "activation_prevalence",
    "population_path": os.environ.get("PILOT_FOLLOW_UPS_POPULATION"),
    "r1_path": os.environ.get("PILOT_FOLLOW_UPS_R1"),
    "variant": os.environ.get("PILOT_FOLLOW_UPS_VARIANT", "r1"),
    "builds_path": os.environ.get("PILOT_FOLLOW_UPS_BUILDS"),
    # Prespecified larger-discovery targets (R1 rule pools hold ~240-330 patients): worst-case prevalence half-widths and minimum frozen-target support.
    "allocation": {"fit_halfwidth": .07, "tune_halfwidth": .10, "min_target_support": 10},
}
if CFG["variant"] not in {"r1", "baseline", "last_3", "all_history"}:
    raise ValueError(f"Unknown variant: {CFG['variant']}")
if MODE == "v0":
    CFG["device"] = "cpu"
OUT = Path(os.environ["RPI_OUT_DIR"]).expanduser().resolve()
if OUT == ROOT or ROOT in OUT.parents:
    raise ValueError("Use a run output directory outside the source checkout")
OUT.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(OUT).free < CFG["min_free_disk_bytes"]:
    raise RuntimeError("Run requires at least 15 GiB free disk")

# pragmatic: small identity/configuration checks use CPU; patient-scale tensors use device.
def canonical_hash(value):
    blob = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return "sha256:" + hashlib.sha256(blob.encode()).hexdigest()

def ordered_rows_hash(record_keys):
    keys = [str(key) for key in record_keys]
    if len(set(keys)) != len(keys):
        raise ValueError("Duplicate record keys")
    return canonical_hash(keys)

IDENTITY_FIELDS = ("source", "data", "ordered_rows", "roles", "predictor_context",
                   "domain", "scaler", "dictionary", "cutoff", "rule", "transform", "config")

def object_identity(parts):
    missing = [key for key in IDENTITY_FIELDS if key not in parts or parts[key] is None]
    if missing:
        raise ValueError(f"Incomplete identity: {missing}")
    return canonical_hash({key: parts[key] for key in IDENTITY_FIELDS})

def file_identity(path):
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return {"path": str(path), "bytes": path.stat().st_size, "method": "sha256",
            "fingerprint": "sha256:" + digest.hexdigest()}

def atomic_json(path, value):
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.")
    try:
        with os.fdopen(fd, "w") as handle:
            json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def git_read(*args):
    return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()

def git_branch():
    result = subprocess.run(["git", "symbolic-ref", "--quiet", "--short", "HEAD"],
                            cwd=ROOT, text=True, capture_output=True)
    return result.stdout.strip() if result.returncode == 0 else None

def set_seed(seed, deterministic):
    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = CFG["tf32"]
    torch.backends.cudnn.allow_tf32 = CFG["tf32"]
    torch.set_num_threads(CFG["cpu_workers"])


In [ ]:
def resolve_device(requested="auto", *, cuda_available=None):
    if requested not in {"auto", "cpu", "cuda"}:
        raise ValueError(f"Unknown device: {requested!r}")
    available = torch.cuda.is_available() if cuda_available is None else cuda_available
    if requested == "cuda" and not available:
        raise RuntimeError("device=cuda requested but CUDA is unavailable")
    if requested == "auto":
        if not available:
            log.warning("device=auto: CUDA unavailable, running on CPU")
        return "cuda" if available else "cpu"
    return requested


In [ ]:
# Report mode renders the saved develop/evaluate exports of every variant: no data, model or scoring work.
REPORT_VARIANTS = ("baseline", "last_3", "all_history")
PALETTE = {"baseline": "#4C72B0", "last_3": "#DD8452", "all_history": "#55A868"}
LAST_OUTCOME = CFG["primary_outcome_last_year"]

def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None

def load_runs(runs):
    runs, loaded = Path(runs).expanduser().resolve(), {}
    for variant in REPORT_VARIANTS:
        develop, evaluate = runs / f"develop-{variant}", runs / f"evaluate-{variant}"
        if not (develop / "freeze.json").is_file():
            continue
        entry = {name: read_json(develop / f"{name}.json") for name in ("freeze", "development", "disappearance", "support", "metrics", "stages")}
        entry["evaluate"] = {int(path.stem.split("-")[1]): read_json(path) for path in sorted(evaluate.glob("evaluate-20*.json"))}
        entry.update(predictor=read_json(evaluate / "predictor.json"), prospective=read_json(evaluate / "prospective.json"),
                     evaluate_metrics=read_json(evaluate / "metrics.json"), evaluate_stages=read_json(evaluate / "stages.json"))
        if entry["evaluate_metrics"] and entry["evaluate_metrics"]["recipe_hash"] != entry["freeze"]["recipe_hash"]:
            raise ValueError(f"{variant}: evaluate recipe differs from its develop freeze")
        loaded[variant] = entry
    if not loaded:
        raise FileNotFoundError(f"No develop/evaluate outputs under {runs}")
    return loaded

def show(frame, name, index=True):
    (OUT / "tables").mkdir(exist_ok=True)
    frame.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    REPORT_INDEX["tables"].append(f"tables/{name}.csv")
    display(frame)

def save(fig, name):
    (OUT / "figures").mkdir(exist_ok=True)
    fig.savefig(OUT / "figures" / f"{name}.png", dpi=120, bbox_inches="tight")
    REPORT_INDEX["figures"].append(f"figures/{name}.png")
    plt.show()
    plt.close(fig)

def heatmap(data, ax, fmt=".2f", size=7, **options):
    """Annotated heatmap; seaborn 0.12 with this matplotlib labels only the first row, so cells are labelled here."""
    sns.heatmap(data, ax=ax, annot=False, **options)
    mesh = ax.collections[0]
    for (i, j), value in np.ndenumerate(np.asarray(data, dtype=float)):
        if np.isfinite(value):
            red, green, blue, _ = mesh.cmap(mesh.norm(value))
            ax.text(j + .5, i + .5, format(value, fmt), ha="center", va="center", fontsize=size,
                    color="black" if .299 * red + .587 * green + .114 * blue > .5 else "white")

def policy_label(policy):
    return f"cap {policy['cap']} · sim {policy['clustering']} · rec {policy['recurrence']}"

def summary(value, key="median"):
    return value.get(key) if isinstance(value, dict) else None

if MODE == "report":
    import matplotlib.pyplot as plt
    import pandas as pd
    import seaborn as sns
    import warnings
    from IPython.display import display
    warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")  # seaborn 0.12 deprecation noise under pandas 2.3
    sns.set_theme(style="whitegrid", context="notebook", font_scale=.9)
    pd.set_option("display.precision", 3, "display.max_columns", 60, "display.width", 240, "display.max_rows", 80)
    RUNS = load_runs(os.environ["PILOT_FOLLOW_UPS_RUNS"])
    REPORT_INDEX = {"variants": list(RUNS), "tables": [], "figures": []}
    ORDER = list(RUNS)
    show(pd.DataFrame([{"variant": variant, "recipe": run["freeze"]["recipe_hash"][7:19], "transform": run["freeze"]["transform"]["kind"],
                        "policy": policy_label(run["freeze"]["policy"]), "allocation": run["freeze"]["allocation"],
                        "develop_hours": run["stages"]["run"]["seconds"] / 3600, "evaluate_hours": run["evaluate_stages"]["run"]["seconds"] / 3600,
                        "references": len(run["evaluate"]), "builds": " ".join(f"{year}:{build}" for year, build in run["metrics"]["builds"].items())}
                       for variant, run in RUNS.items()]).set_index("variant"), "overview")

## Focused invariants

Checks cover device failure/fallback, ordered row identity, and complete object/cache identity. Later sections extend this same small check area as their calculations arrive.


## Patient support and protected roles

Use original reference-only splits, preserving each patient's role across years.
Cached R1 objects open read-only after exact source/config/population verification.
No real inputs are read by v0. Future scientific stages call `open_r1_inputs()`
and `reference_support_audit()`; redesigned protected-role outcomes stay masked
until frozen evaluation. 2015 outcomes stay masked in every mode.

No external reserve is presumed safe. A separate, single within-discovery
candidate can enlarge the fit half while retaining independent tuning patients.
Its precision targets must be frozen before scoring; the sample-size bound is
for prevalence, and does not promise precision for rare-positive rule fidelity.
Frozen activation positive/negative support is checked before adopting it.


In [ ]:
R1_CONFIG_HASH = "c71b8d1d7d28bebb6947317fefc0b9fe4891e937ba97e356b9745f929e3e7b02"
R1_SOURCE_HASH = "fe94560ebbfd3413f63eca78500827d85b47e9667664dd927a31b120bf87ac63"
# UnitCache records hash the legacy .py and .ipynb together, as the R1 notebook did.
R1_CACHE_SOURCE_HASH = "8a71a6759b29e3b967c64cd5cd5e696a47a13a18cfe4112791dc0e229d9231d7"
R1_NOTEBOOK_HASH = "b91ed414df875ab5605bb340caa93db1784e1fae09f7d81c48deecd3250904f9"
R1_COMMIT = "d73a657"
POPULATION_HASH = "c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7"
# Plan v3 legacy edits: context window (rows and calendar domains), reference start, SAE identity report, checks.
LEGACY_CHANGES = {"resolve_config", "configuration_overrides", "_predictor_unit", "encode_reference_dictionary",
                  "fit_frozen_predictor", "capture_frozen_predictions", "predictor_unit",
                  "_sae_state_difference", "_predictor_unit_checks", "=CONTEXT_WINDOWS", "=FIRST_CONTEXT_YEAR"}
WINDOWS = {"baseline": "reference_only", "last_3": "last_3", "all_history": "all_history"}
VARIANT_KEYS = {"context_window", "reference_year_candidates", "predictor_cpu_from", "config_hash", "profile", "device", "data_path",
                "cache_dir", "output_dir", "disk_guard_gib", "joblib_workers", "inference_batch_rows"}
PROTECTED = {"tabpfn_context", "sae_discovery", "cav_fit", "concept_selection", "evaluation"}

def patient_partition(groups):
    seen = set()
    for name, members in groups.items():
        ids = {str(patient) for patient in members}
        if seen & ids:
            raise ValueError(f"Patient roles overlap: {name}")
        seen |= ids
    return seen

def permanent_role_audit(patients, years, roles, reference):
    assigned = {}
    for patient, year, role in zip(patients, years, roles, strict=True):
        if year < reference:
            if role:
                raise ValueError("Pre-reference patient received a role")
            continue
        if not role or assigned.setdefault(str(patient), role) != role:
            raise ValueError("Missing or changing permanent patient role")
    return assigned

def support_rows(population, roles, reference, *, frozen_evaluation=False, targets=None):
    patient_partition(roles["reference_roles"])
    permanent_role_audit(population["patient_id"], population["year"], roles["row_role"], reference)
    if targets is not None and (np.asarray(targets).shape != population["year"].shape or not np.isin(targets, [0, 1]).all()):
        raise ValueError("Frozen targets must be aligned binary masks")
    result = []
    for year in sorted(set(population["year"])):
        for role in sorted(set(roles["row_role"]) - {""}):
            mask = (population["year"] == year) & (roles["row_role"] == role)
            n = int(mask.sum())
            if not n:
                continue
            allowed = year <= 2014 and (role not in PROTECTED or frozen_evaluation)
            deaths = int(population["y"][mask].sum()) if allowed else None
            result.append({"reference": reference, "year": int(year), "role": role,
                "patients": len(set(population["patient_id"][mask])), "rows": n,
                "entrants": int(roles["is_entrant"][mask].sum()), "deaths": deaths,
                "survivors": n - deaths if deaths is not None else None,
                "outcome_status": "available" if allowed else "invalid_2015" if year > 2014 else "protected",
                "activation_positives": int(np.asarray(targets)[mask].sum()) if targets is not None else None,
                "activation_status": "supplied frozen target" if targets is not None else "pending frozen target"})
    return result

def worst_case_halfwidth(n):
    return 1.96 / (2 * np.sqrt(n + 1.96**2)) if n else 0.5

def discovery_candidate(reference_roles, existing_fit, *, fit_halfwidth, tune_halfwidth):
    patient_partition(reference_roles)
    pool, fit = set(map(str, reference_roles["rule_discovery"])), set(map(str, existing_fit))
    if not fit <= pool or not 0 < fit_halfwidth < .5 or not 0 < tune_halfwidth < .5:
        raise ValueError("Invalid discovery pool or precision targets")
    if worst_case_halfwidth(len(fit)) <= fit_halfwidth and worst_case_halfwidth(len(pool - fit)) <= tune_halfwidth:
        return {"status": "baseline_sufficient", "fit": sorted(fit), "tune": sorted(pool - fit)}
    # A single nested candidate: preserve original fit; move only rule-tuning patients.
    donors = sorted(pool - fit, key=lambda p: canonical_hash([CFG["split_seed"], p]))
    for count in range(1, len(donors) + 1):
        larger = fit | set(donors[:count])
        if worst_case_halfwidth(len(larger)) <= fit_halfwidth and worst_case_halfwidth(len(pool - larger)) <= tune_halfwidth:
            return {"status": "candidate", "fit": sorted(larger), "tune": sorted(pool - larger),
                    "selection": "smallest deterministic expansion meeting prevalence precision",
                    "requires": "refit rules and validate frozen-target positive/negative support before adoption"}
    return {"status": "unavailable", "reason": "insufficient independent tuning support",
            "external_reserve": "unknown: no supplied upstream/evaluation provenance"}

def allocation_rows(patients, rule, fit, tune):
    """One support-led larger fit inside rule_discovery; both allocations score on the remaining common tune patients."""
    targets = {key: CFG["allocation"][key] for key in ("fit_halfwidth", "tune_halfwidth")}
    candidate = discovery_candidate({"rule_discovery": patients[rule]}, patients[fit], **targets)
    if candidate["status"] != "candidate":
        return None, None, {key: value for key, value in candidate.items() if key not in {"fit", "tune"}}
    larger, common = rule[np.isin(patients[rule], candidate["fit"])], rule[np.isin(patients[rule], candidate["tune"])]
    return larger, common, {"status": "candidate", "fit": [len(fit), len(larger)], "common_tune": len(common), "original_tune": len(tune)}

def legacy_changes():
    """Top-level legacy definitions changed since the pinned R1 source; R1 loaders must stay untouched."""
    import ast
    original = subprocess.check_output(["git", "show", f"{R1_COMMIT}:concept_subgroup_drift_pilot.py"], cwd=ROOT)
    if hashlib.sha256(original).hexdigest() != R1_SOURCE_HASH:
        raise ValueError("Pinned R1 legacy source differs from its recorded hash")
    if file_identity(ROOT / "concept_subgroup_drift_pilot.ipynb")["fingerprint"] != "sha256:" + R1_NOTEBOOK_HASH:
        raise ValueError("Legacy pilot notebook differs from R1")
    def parts(text):
        named = {}
        for node in ast.parse(text).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                key = node.name
            elif isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):
                key = "=" + node.targets[0].id
            else:
                key = ast.get_source_segment(text, node)
            named[key] = ast.get_source_segment(text, node)
        return named
    old, new = parts(original.decode()), parts((ROOT / "concept_subgroup_drift_pilot.py").read_text())
    changed = sorted(key for key in old.keys() | new.keys() if old.get(key) != new.get(key))
    if not set(changed) <= LEGACY_CHANGES:
        raise ValueError(f"Legacy edits outside plan v3: {sorted(set(changed) - LEGACY_CHANGES)}")
    return changed

def legacy_source_hash():
    names = ("concept_subgroup_drift_pilot.py", "concept_subgroup_drift_pilot.ipynb")
    sources = {name: file_identity(ROOT / name)["fingerprint"].split(":")[1] for name in names}
    return hashlib.sha256(json.dumps(sources, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def build_for(variant, reference):
    """Build holding a variant's objects for one reference; identical effective context years share one build."""
    if variant == "r1" or reference == 2007 or (variant == "baseline" and reference <= 2013):
        return "r1"
    if variant == "last_3" and reference in (2008, 2009):
        return "all_history"
    return variant

def open_inputs():
    # Narrow compatibility imports: checksum loading, immutable caches, original patient split.
    from concept_subgroup_drift_pilot import UnitCache, load_pilot_population
    changed = legacy_changes()
    if CFG["variant"] != "r1" and not CFG["builds_path"]:
        raise ValueError("PILOT_FOLLOW_UPS_BUILDS is required for window/baseline variants")
    r1 = Path(CFG["r1_path"]).expanduser() if CFG["r1_path"] else ROOT.parent / "rpi-runs/SAE-TCAV/r1-pilot-fe945-20261009"
    population_path = Path(CFG["population_path"]).expanduser() if CFG["population_path"] else ROOT / "stats/temporal_robustness/_population/prepared.pkl"
    r1_config = json.loads((r1 / "out/resolved-config.json").read_text())
    shared = lambda config: {key: value for key, value in config.items() if key not in VARIANT_KEYS}
    caches, configs, builds = {}, {}, {}
    for name in sorted({build_for(CFG["variant"], r) for r in REFERENCES}):
        if name == "r1":
            path, config, source = r1, r1_config, R1_CACHE_SOURCE_HASH
            cache = UnitCache(r1 / "unit-cache", config, source, expected_hash=R1_CONFIG_HASH, read_only=True)
        else:
            path = Path(CFG["builds_path"]).expanduser() / name
            config, source = json.loads((path / "out/resolved-config.json").read_text()), legacy_source_hash()
            if config.get("context_window", "reference_only") != WINDOWS[name] or shared(config) != shared(r1_config):
                raise ValueError(f"Build {name} differs from R1 beyond its context window and reference years")
            cache = UnitCache(path / "unit-cache", config, source, expected_hash=config["config_hash"], read_only=True)
        caches[name], configs[name] = cache, config
        builds[name] = {"path": str(path), "config_hash": config["config_hash"], "source_hash": source,
                        "context_window": config.get("context_window", "reference_only"), "references": config["reference_year_candidates"]}
    population = load_pilot_population(population_path, POPULATION_HASH)
    return population, caches, configs, {"variant": CFG["variant"], "builds": builds, "legacy_changes": changed}

def reference_support_audit(population, cache, config, reference, *, frozen_evaluation=False):
    from concept_subgroup_drift_pilot import reference_year_roles
    roles = reference_year_roles(population, reference, seed=config["split_seed"])
    predictor = cache.load(f"P/{reference}")
    if predictor["roles_identity"] != roles["identity"] or predictor["population_identity"] != population["fingerprint"]["population_identity"]:
        raise ValueError("Cached predictor role/population identity mismatch")
    table = support_rows(population, roles, reference, frozen_evaluation=frozen_evaluation)
    return {"support": table, "roles": roles, "allocation": {
        "original": {name: len(ids) for name, ids in roles["reference_roles"].items()},
        "larger_pool": "unavailable without qualified external reserve; protected roles retained",
        "inner_reallocation": "pending frozen activation support and prespecified precision targets",
        "evidence": "R1 patients previously examined; redesigned evaluation remains protected"}}


In [ ]:
def focused_role_checks():
    assert patient_partition({"fit": ["a"], "tune": ["b"]}) == {"a", "b"}
    for call in [lambda: patient_partition({"fit": ["a"], "evaluation": ["a"]}),
                 lambda: permanent_role_audit(["a", "a"], [2007, 2008], ["fit", "evaluation"], 2007),
                 lambda: permanent_role_audit(["a"], [2006], ["fit"], 2007)]:
        try:
            call()
        except ValueError:
            pass
        else:
            raise AssertionError("Patient leakage must fail")
    assert permanent_role_audit(["new", "new"], [2008, 2009], ["fitting", "fitting"], 2007) == {"new": "fitting"}
    population = {"patient_id": np.array(["a", "b"]), "year": np.array([2007, 2015]), "y": np.array([1, 1])}
    roles = {"reference_roles": {"evaluation": ["a"]}, "row_role": np.array(["evaluation", "evaluation"]), "is_entrant": np.array([False, True])}
    table = support_rows(population, roles, 2007)
    assert [row["deaths"] for row in table] == [None, None]
    table = support_rows(population, roles, 2007, frozen_evaluation=True)
    assert [row["deaths"] for row in table] == [1, None]
    patients = [str(i) for i in range(600)]
    reference_roles = {"rule_discovery": patients, "evaluation": ["protected"]}
    candidate = discovery_candidate(reference_roles, patients[:300], fit_halfwidth=.05, tune_halfwidth=.1)
    assert candidate["status"] == "candidate" and len(candidate["fit"]) > 300
    assert patient_partition({"fit": candidate["fit"], "tune": candidate["tune"]}) == set(patients)
    assert "protected" not in candidate["fit"]
    assert discovery_candidate(reference_roles, patients[:300], fit_halfwidth=.1, tune_halfwidth=.1)["status"] == "baseline_sufficient"
    assert discovery_candidate({"rule_discovery": ["a", "b"]}, ["a"], fit_halfwidth=.05, tune_halfwidth=.1)["status"] == "unavailable"
    rule = np.arange(320)
    larger, common, audit = allocation_rows(np.array(patients), rule, rule[:160], rule[160:])
    assert set(rule[:160]) < set(larger) and not set(larger) & set(common) and set(common) < set(rule[160:])
    assert audit["fit"] == [160, len(larger)] and audit["common_tune"] == len(common) == 320 - len(larger)
    assert allocation_rows(np.array(patients), rule[:237], rule[:118], rule[118:237])[0] is None
    assert [build_for("last_3", r) for r in (2007, 2008, 2009, 2010)] == ["r1", "all_history", "all_history", "last_3"]
    assert [build_for("baseline", r) for r in (2013, 2014)] == ["r1", "baseline"] and build_for("all_history", 2007) == "r1"
    changed = legacy_changes()
    return {"status": "passed", "legacy_changes": changed,
            "scope": "role separation/permanence/outcome protection/discovery candidate/allocation/aliases/legacy inventory"}


In [ ]:
# Report: patient roles and support. Roles are patient-level, so every variant shares them; checked below.
def support_frame(run):
    return pd.DataFrame([row for value in run["support"].values() for row in value["support"]])

def show_support():
    frames = {variant: support_frame(run) for variant, run in RUNS.items()}
    frame = frames[ORDER[0]]
    shared = all(other[["reference", "year", "role", "patients"]].equals(frame[["reference", "year", "role", "patients"]]) for other in frames.values())
    print("Support identical across variants:", shared)
    show(frame[frame.year == frame.reference].pivot_table(index="reference", columns="role", values="patients", aggfunc="sum"), "support_roles_at_reference")
    fig, axes = plt.subplots(1, 4, figsize=(21, 4.6), gridspec_kw={"width_ratios": [1, 1, 1, 1.1]})
    for ax, role in zip(axes, ("evaluation", "rule_discovery", "fitting")):
        grid = frame[frame.role == role].pivot_table(index="reference", columns="year", values="patients")
        heatmap(grid, ax, fmt=".0f", cmap="Blues", cbar=False)
        ax.set_title(f"{role} patients (reference × year)")
    evaluation = frame[frame.role == "evaluation"].assign(entrant_share=lambda d: d.entrants / d.patients.where(d.patients > 0))
    sns.lineplot(evaluation, x="year", y="entrant_share", hue="reference", palette="viridis", marker="o", ax=axes[3], legend="brief")
    axes[3].set(title="Entrant share of evaluation patients", ylim=(0, 1))
    axes[3].legend(fontsize=7, ncol=2, title="reference")
    save(fig, "support")
    # Frozen activation targets: positives on reference fitting patients per member.
    cutoffs = pd.DataFrame([{"variant": variant, "reference": int(reference), "medoid": member["medoid"],
                             "positive_fit": member["cutoff_audit"].get("positive_fit"), "fraction": member["cutoff_audit"].get("fraction"),
                             "ties": member["cutoff_audit"].get("ties"), "status": member["cutoff_audit"].get("status")}
                            for variant, run in RUNS.items() for reference, members in run["development"].items() for member in members])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.boxplot(cutoffs, x="reference", y="positive_fit", hue="variant", palette=PALETTE, ax=axes[0], fliersize=2)
    axes[0].set_title("Frozen-target positives on fitting patients, per member")
    sns.countplot(cutoffs[cutoffs.variant == ORDER[0]], x="reference", hue="fraction", palette="crest", ax=axes[1])
    axes[1].set_title("Cutoff positive-fraction chosen per member")
    save(fig, "support_targets")

if MODE == "report":
    show_support()

## Reference explanations and continuous activation

Hold predictor, dictionary, target and patients fixed while comparing counts,
presence and, only with clinical definition/source, one aggregate representation.
`fit_member_explanation` exposes generation → cap → family clustering → outer
bootstrap recurrence → rule selection. Supply baseline plus at most three
single-policy variants from reference development. Returned tune fidelity is
development evidence; final patient fidelity comes from the frozen final suite.
Mortality is excluded from explanatory features. Continuous comparisons retain
both mean/MSE and median/MAE constants, ridge and one small nonlinear model.
Scalers and high-activation cutoffs fit on reference fitting patients only.


In [ ]:
def explanation_features(X, feature_names, kind="counts", aggregate=None):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] != len(feature_names) or not np.isfinite(X).all() or (X < 0).any():
        raise ValueError("Expected finite nonnegative event-count features")
    if any(str(name).upper() == "DEATH" for name in feature_names):
        raise ValueError("Mortality cannot be an explanatory feature")
    if kind == "counts":
        return X.copy(), list(feature_names)
    if kind == "presence":
        return (X > 0).astype(float), list(feature_names)
    if kind != "aggregate" or not aggregate or not aggregate.get("clinical_justification") or not aggregate.get("source"):
        raise ValueError("Aggregation unavailable without recorded clinical definition and source")
    groups = aggregate["groups"]
    indices = [[feature_names.index(name) for name in names] for names in groups.values()]
    if any(not group for group in indices):
        raise ValueError("Empty aggregate definition")
    return np.column_stack([X[:, group].sum(axis=1) for group in indices]), list(groups)

def development_rows(patients, fit_rows, tune_rows, allowed_patients):
    fit, tune = np.asarray(fit_rows, dtype=int), np.asarray(tune_rows, dtype=int)
    if not len(fit) or not len(tune) or min(fit.min(), tune.min()) < 0 or max(fit.max(), tune.max()) >= len(patients):
        raise ValueError("Nonempty valid reference fit/tune rows required")
    patient_partition({"fit": np.asarray(patients)[fit], "tune": np.asarray(patients)[tune]})
    if not set(map(str, np.asarray(patients)[np.r_[fit, tune]])) <= set(map(str, allowed_patients)):
        raise ValueError("Development includes protected or future patients")
    return fit, tune

def candidate_policies(baseline, changes=()):
    allowed = {"cap", "clustering", "recurrence"}
    if set(baseline) != allowed or len(changes) > 3 or any(len(change) != 1 or not set(change) <= allowed for change in changes):
        raise ValueError("At most three single-policy changes required")
    policies = [dict(baseline)] + [dict(baseline, **change) for change in changes]
    for policy in policies:
        if type(policy["cap"]) is not int or policy["cap"] < 1 or not 0 <= policy["clustering"] <= 1 or not 0 < policy["recurrence"] <= 1:
            raise ValueError("Invalid candidate policy")
    return policies

def outer_recurrence(bootstrap_ids, total_bootstraps):
    if total_bootstraps < 1 or any(not 0 <= value < total_bootstraps for value in bootstrap_ids):
        raise ValueError("Invalid outer bootstrap identity")
    return len(set(bootstrap_ids)) / total_bootstraps

def fidelity(target, selected):
    from semantic_rules import binary_metrics
    result = binary_metrics(np.asarray(target, dtype=bool), np.asarray(selected, dtype=bool)).to_dict()
    result.update(false_positive=result["n_selected"] - result["true_positive"], false_negative=result["n_positive"] - result["true_positive"])
    result["status"] = "available" if result["n_positive"] else "no activation positives" if result["n_samples"] else "empty cohort"
    if not result["n_selected"]:
        result["precision"] = None
    if not result["n_positive"]:
        for metric in ("recall", "f2", "lift"):
            result[metric] = None
    if not result["n_positive"] and not result["n_selected"]:
        result["jaccard"] = None
    if not result["n_samples"]:
        for metric in ("precision", "recall", "f2", "lift", "prevalence", "coverage", "wracc", "jaccard"):
            result[metric] = None
    return result


def cap_by_meaning(occurrences, maximum, X_fit):
    """Per bootstrap top-N as semantic_experiment._cap_occurrences, but duplicates are rules selecting
    the same fit rows per condition: thresholds between the same adjacent observed fit values are one rule.
    Rule IDs carry the random threshold, so ID dedup kept e.g. presence 'x > .31' and 'x > .74' apart."""
    columns = [np.unique(column) for column in np.asarray(X_fit, dtype=float).T]

    def meaning(rule):
        return tuple((c.feature_index, c.operator, int(np.searchsorted(columns[c.feature_index], c.threshold, side="right")))
                     for c in rule.conditions)
    grouped = {}
    for occurrence in occurrences:
        grouped.setdefault(occurrence.bootstrap_id, []).append(occurrence)
    retained = []
    for bootstrap_id in sorted(grouped):
        ranked = sorted(grouped[bootstrap_id], key=lambda item: (-item.oob_recall, -item.oob_precision, -item.fit_recall,
            -item.fit_precision, item.rule.length, item.rule.rule_id, item.tree_id, item.leaf_id))
        seen = set()
        for occurrence in ranked:
            key = meaning(occurrence.rule)
            if key not in seen and len(seen) < maximum:
                seen.add(key)
                retained.append(occurrence)
    return tuple(retained)

def fit_member_explanation(X, names, activations, cutoff, patients, fit_rows, tune_rows,
                           allowed_patients, backend_config, selection_config, policies):
    from dataclasses import asdict
    from stable_rule_backend import StableRuleBackendConfig, discover_stable_rule_candidates
    from semantic_compare import RuleSimilarityConfig, cluster_rule_families, recurrent_representatives
    from semantic_rules import RuleSetSelectionConfig, select_rule_set
    if not policies:
        raise ValueError("Baseline policy required")
    candidate_policies(policies[0], [{key: value for key, value in policy.items() if value != policies[0].get(key)} for policy in policies[1:]])
    fit, tune = development_rows(patients, fit_rows, tune_rows, allowed_patients)
    X, activations = np.asarray(X), np.asarray(activations, dtype=float)
    if activations.shape != (len(X),) or not np.isfinite(activations).all() or not np.isfinite(cutoff):
        raise ValueError("Aligned frozen activations and finite cutoff required")
    target = activations > cutoff  # Same frozen activation target for every policy/feature comparison.
    discovery = discover_stable_rule_candidates(X[fit], target[fit].astype(int), names,
        groups=np.asarray(patients)[fit], config=StableRuleBackendConfig(**backend_config))
    results = []
    for policy in policies:
        capped = cap_by_meaning(discovery.occurrences, policy["cap"], X[fit])
        clustered = cluster_rule_families(capped, X[tune], RuleSimilarityConfig(
            similarity_threshold=policy["clustering"], min_recurrence=policy["recurrence"]),
            total_bootstraps=backend_config["n_bootstraps"])
        offered = recurrent_representatives(clustered, policy["recurrence"])
        selected = select_rule_set(offered, X[tune], target[tune], RuleSetSelectionConfig(**selection_config))
        family_rows = [{"family": family.family_id, "occurrences": family.occurrence_count,
            "outer_recurrence": outer_recurrence(family.bootstrap_ids, backend_config["n_bootstraps"]),
            "reason": "offered" if family.retained else "below recurrence"} for family in clustered.families]
        diagnostics = [asdict(item) for item in discovery.bootstrap_diagnostics]
        results.append({"policy": policy, "backend": backend_config, "selection": selection_config,
            "generated_occurrences": len(discovery.occurrences), "unique_rules": len(discovery.candidates),
            "capped_occurrences": len(capped), "removed_by_cap_or_duplicate": len(discovery.occurrences) - len(capped),
            "duplicate_rule": "same fit rows per condition",
            "families": family_rows, "offered": len(offered), "bootstrap_diagnostics": diagnostics,
            "selection_reason": "selected" if selected.feasible else selected.reason,
            "rule": selected.rule_set, "rule_dict": selected.rule_set.to_dict(),
            "tune": fidelity(target[tune], selected.rule_set.mask(X[tune])),
            "complexity": {"rules": len(selected.rule_set.rules),
                           "conditions": sum(rule.length for rule in selected.rule_set.rules)},
            "cutoff": float(cutoff), "target_identity": canonical_hash(target.tolist()),
            "fit_identity": ordered_rows_hash(np.asarray(patients)[fit]),
            "tune_identity": ordered_rows_hash(np.asarray(patients)[tune]),
            "evidence": "reference development only; final patient fidelity not scored"})
    return results

def activation_errors(truth, predicted, high_cutoff):
    truth, predicted = np.asarray(truth), np.asarray(predicted)
    result = {}
    for name, mask in {"full": np.ones(len(truth), dtype=bool), "zero": truth == 0,
                       "positive": truth > 0, "high": truth >= high_cutoff}.items():
        error = predicted[mask] - truth[mask]
        result[name] = {"n": len(error), "mae": float(np.abs(error).mean()) if len(error) else None,
                        "mse": float((error**2).mean()) if len(error) else None}
    return result

def fit_continuous_activation(X, activations, patients, fit_rows, tune_rows, allowed_patients):
    from sklearn.dummy import DummyRegressor
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import HistGradientBoostingRegressor
    from threadpoolctl import threadpool_limits
    fit, tune = development_rows(patients, fit_rows, tune_rows, allowed_patients)
    X, y = np.asarray(X), np.asarray(activations, dtype=float)
    if y.shape != (len(X),) or not np.isfinite(y).all() or (y < 0).any():
        raise ValueError("Finite aligned nonnegative activations required")
    models = {"constant_mean": DummyRegressor(strategy="mean"), "constant_median": DummyRegressor(strategy="median"),
        "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "nonlinear": HistGradientBoostingRegressor(max_iter=50, max_leaf_nodes=7,
            min_samples_leaf=10, l2_regularization=1.0, early_stopping=False, random_state=CFG["seed"])}
    high_cutoff = float(np.quantile(y[fit][y[fit] > 0], .9)) if (y[fit] > 0).any() else float("inf")
    scores = {}
    # pragmatic: tiny tabular ridge/rule models are CPU work; bound BLAS threads to four.
    with threadpool_limits(limits=CFG["cpu_workers"]):
        for name, model in models.items():
            model.fit(X[fit], y[fit])
            scores[name] = activation_errors(y[tune], model.predict(X[tune]), high_cutoff)
    return {"models": models, "scores": scores, "high_cutoff": high_cutoff if np.isfinite(high_cutoff) else None,
            "high_status": "available" if np.isfinite(high_cutoff) else "no positive fitting activations",
            "settings": {name: {key: repr(value) for key, value in model.get_params().items()} for name, model in models.items()},
            "evidence": "reference patient-separated tune; no final evaluation"}


In [ ]:
def focused_reference_checks():
    assert outer_recurrence([0, 0, 0], 2) == .5
    assert outer_recurrence([0, 1, 1], 2) == 1
    X = np.array([[0., 2.], [3., 0.], [1., 4.]])
    transformed, names = explanation_features(X, ["event_a", "event_b"], "presence")
    assert np.array_equal(transformed, [[0, 1], [1, 0], [1, 1]]) and X[0, 1] == 2
    assert len(candidate_policies({"cap": 3, "clustering": .7, "recurrence": .5}, [{"cap": 4}])) == 2
    for call in [lambda: development_rows(["a", "b"], [0], [1], ["a"]),
                 lambda: development_rows(["a", "a"], [0], [1], ["a"]),
                 lambda: explanation_features(X, ["event_a", "event_b"], "aggregate")]:
        try:
            call()
        except ValueError:
            pass
        else:
            raise AssertionError("Unsafe development/undefined aggregation must fail")
    rng = np.random.default_rng(42)
    X = rng.normal(size=(40, 2))
    y = np.maximum(X[:, 0], 0)
    ids = np.array([f"synthetic-{i}" for i in range(40)])
    result = fit_continuous_activation(X, y, ids, np.arange(25), np.arange(25, 40), ids)
    assert set(result["scores"]) == {"constant_mean", "constant_median", "ridge", "nonlinear"}
    assert all(np.isfinite(value["full"]["mae"]) for value in result["scores"].values())
    from types import SimpleNamespace as N
    def occurrence(boot, threshold, recall):
        condition = N(feature_index=0, operator=">", threshold=threshold)
        return N(bootstrap_id=boot, oob_recall=recall, oob_precision=0, fit_recall=0, fit_precision=0, tree_id=0, leaf_id=0,
                 rule=N(conditions=(condition,), length=1, rule_id=f"r{threshold}"))
    binary = np.array([[0.], [1.], [1.]])
    kept = cap_by_meaning([occurrence(0, .31, .9), occurrence(0, .74, .8), occurrence(0, 1.5, .1), occurrence(1, .5, .2)], 2, binary)
    assert [item.rule.rule_id for item in kept] == ["r0.31", "r1.5", "r0.5"], [item.rule.rule_id for item in kept]
    policies = candidate_policies({"cap": 3, "clustering": .7, "recurrence": .5})
    rules = fit_member_explanation(X, ["a", "b"], y, .25, ids, np.arange(25), np.arange(25, 40), ids,
        {"n_bootstraps": 2, "trees_per_bootstrap": 2, "max_depth": 2, "min_samples_leaf": 2},
        {"min_precision": 0., "min_lift": 0., "min_marginal_recall": 0.}, policies)
    assert len(rules) == 1 and rules[0]["tune"]["n_samples"] == 15
    return {"status": "passed", "scope": "outer recurrence/feature transform/reference-only support/tiny continuous models"}


In [ ]:
# Report: reference explanation development (tune patients only), policies, representations, allocation, thresholds.
def development_frame():
    rows, families, features = [], [], []
    for variant, run in RUNS.items():
        frozen = (run["freeze"]["transform"]["kind"], policy_label(run["freeze"]["policy"]))
        for reference, members in run["development"].items():
            for member in members:
                for result in member.get("results", []):
                    kind = result["transform"]["kind"]
                    for item in result["policies"]:
                        setting = policy_label(item["policy"])
                        rows.append({"variant": variant, "reference": int(reference), "family": member["family"], "transform": kind,
                                     "setting": setting, "generated": item["generated_occurrences"], "capped": item["capped_occurrences"],
                                     "families": len(item["families"]), "offered": item["offered"], "selected": item["selection_reason"] == "selected",
                                     "rules": item["complexity"]["rules"], "conditions": item["complexity"]["conditions"]})
                        if setting == policy_label(run["freeze"]["policy"]):
                            families += [{"variant": variant, "transform": kind, "outer_recurrence": family["outer_recurrence"],
                                          "occurrences": family["occurrences"]} for family in item["families"]]
                        if (kind, setting) == frozen and item["selection_reason"] == "selected":
                            features += [{"variant": variant, "reference": int(reference), "feature": condition["feature_name"], "operator": condition["operator"]}
                                         for rule in item["rule_dict"]["rules"] for condition in rule["conditions"]]
    return pd.DataFrame(rows), pd.DataFrame(families), pd.DataFrame(features)

def show_explanations():
    table = pd.DataFrame([{"variant": variant, "transform": row["setting"]["transform"]["kind"], "setting": policy_label(row["setting"]["policy"]),
                           "median_tune_f2": row["median_tune_f2"], "members": row["n"]}
                          for variant, run in RUNS.items() for row in run["freeze"]["development_table"]])
    show(table.pivot_table(index=["transform", "setting"], columns="variant", values="median_tune_f2")[ORDER], "development_median_tune_f2")
    rows, families, features = development_frame()
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.4), sharey=True)
    for ax, kind in zip(axes, ("counts", "presence")):
        sns.barplot(table[table["transform"] == kind], x="setting", y="median_tune_f2", hue="variant", palette=PALETTE, hue_order=ORDER, ax=ax)
        ax.set(title=f"{kind}: median tune F2 per setting (members)", xlabel="", ylim=(0, 1))
        ax.tick_params(axis="x", rotation=15)
    save(fig, "development_f2")
    fig, axes = plt.subplots(1, 3, figsize=(20, 4.4))
    funnel = rows.melt(id_vars=["variant", "transform", "setting"], value_vars=["generated", "capped", "families", "offered"], var_name="stage", value_name="n")
    sns.barplot(funnel[funnel.setting == policy_label(RUNS[ORDER[0]]["freeze"]["policy"])], x="stage", y="n", hue="transform", estimator="median",
                errorbar=("pi", 50), ax=axes[0])
    axes[0].set(yscale="symlog", title="Rule pipeline, frozen policy (median, IQR)")
    selected = rows.groupby(["variant", "transform", "setting"]).selected.mean().reset_index()
    sns.barplot(selected, x="setting", y="selected", hue="transform", ax=axes[1], errorbar=None)
    axes[1].set(title="Share of members with a selected rule (all variants)", xlabel="", ylim=(0, 1))
    axes[1].tick_params(axis="x", rotation=15)
    sns.histplot(families, x="outer_recurrence", hue="transform", element="step", stat="density", common_norm=False, bins=30, ax=axes[2])
    axes[2].set(title="Family outer recurrence, frozen policy (all variants)")
    save(fig, "development_pipeline")
    show(rows.groupby(["transform", "setting", "variant"])[["generated", "capped", "families", "offered", "selected", "conditions"]].median()
         .assign(selected_share=rows.groupby(["transform", "setting", "variant"]).selected.mean()).unstack("variant"), "development_pipeline")
    if not features.empty:
        top = features.groupby("feature").size().nlargest(20).index
        fig, ax = plt.subplots(figsize=(11, 6))
        sns.countplot(features[features.feature.isin(top)], y="feature", hue="variant", order=top, palette=PALETTE, hue_order=ORDER, ax=ax)
        ax.set(title="Features in selected frozen-recipe rules (conditions, all references)", ylabel="")
        save(fig, "development_rule_features")
    continuous = pd.DataFrame([{"variant": variant, "reference": int(reference), "model": model, "subset": subset, "mae": score["mae"], "n": score["n"]}
                               for variant, run in RUNS.items() for reference, members in run["development"].items() for member in members
                               if member.get("continuous") for model, scores in member["continuous"]["scores"].items()
                               for subset, score in scores.items() if score["mae"] is not None])
    if not continuous.empty:
        grid = sns.catplot(continuous, x="subset", y="mae", hue="model", col="variant", col_order=ORDER, kind="box", height=3.8, aspect=1.2,
                           order=["full", "zero", "positive", "high"], fliersize=2)
        grid.set(yscale="log")
        grid.figure.suptitle("Continuous activation models on reference tune patients (MAE)", y=1.03)
        save(grid.figure, "development_continuous")
    allocation = pd.DataFrame([{"variant": variant, "reference": int(reference), "status": member["allocation"]["status"]}
                               for variant, run in RUNS.items() for reference, members in run["development"].items() for member in members
                               if member.get("allocation")])
    counts = allocation.pivot_table(index="status", columns="variant", aggfunc="size", fill_value=0)
    targets = pd.DataFrame({variant: run["freeze"]["allocation_table"] for variant, run in RUNS.items()})
    show(pd.concat([counts, targets]).reindex(columns=ORDER), "allocation")
    thresholds = pd.DataFrame([{"variant": variant, "reference": int(year), **{key: value.get(key) for key in ("threshold", "death_f1", "precision", "recall", "deaths", "rows")}}
                               for variant, run in RUNS.items() for year, value in run["freeze"]["thresholds"].items() if value.get("status") == "available"])
    fig, axes = plt.subplots(1, 4, figsize=(21, 3.8))
    for ax, key in zip(axes, ("threshold", "death_f1", "precision", "recall")):
        sns.lineplot(thresholds, x="reference", y=key, hue="variant", palette=PALETTE, hue_order=ORDER, marker="o", ax=ax, legend=key == "threshold")
        ax.set_title(f"Frozen Death-F1 threshold: {key}")
    save(fig, "thresholds")
    show(thresholds.pivot_table(index="reference", columns="variant", values=["threshold", "death_f1"]), "thresholds")

if MODE == "report":
    show_explanations()

## Member and local-group correspondence

Fit three separate descriptions with the preceding member-fitting calculation;
freeze each positive-activation cutoff at the same fraction, then score the full
directed 3×3 matrix on identical independent patients. A common union-rule is a
separate tune-only alternative and needs its own final validation. Continuous
evidence and paired intervals precede any categorical interpretation.

Groups contain at most five factors. Tune rank 1–3 and small ridge penalties
using reference patients, then compare both mapping directions against constant,
shuffled and reference-matched non-family controls. Raw contributions multiply
decoder directions by the reference scale. Remove/substitute them while keeping
the original residual and bias; head replay and Brier effects are scored later.
Signed/absolute cosine and local subspaces are separate from activation Pearson
matching; full dictionary spans never establish local signal agreement.


In [ ]:
def patient_intervals(patients, statistic, *, replicates=500, seed=None):
    patients = np.asarray(patients).astype(str)
    unique = np.unique(patients)
    if len(unique) < 2:
        return {"status": "insufficient independent patients", "patients": len(unique), "intervals": None}
    if replicates < 2:
        raise ValueError("At least two patient bootstrap replicates required")
    clusters = {patient: np.flatnonzero(patients == patient) for patient in unique}
    rng = np.random.default_rng(CFG["seed"] if seed is None else seed)
    draws = np.asarray([statistic(np.concatenate([clusters[p] for p in rng.choice(unique, len(unique), replace=True)])) for _ in range(replicates)])
    bounds = []
    for column in draws.T:
        finite = column[np.isfinite(column)]
        bounds.append(np.quantile(finite, [.025, .975]).tolist() if len(finite) >= 2 else None)
    return {"status": "available", "patients": len(unique), "replicates": replicates,
            "method": "paired patient-cluster percentile", "intervals": bounds}

def member_cutoffs(activations, fitting_rows, fraction):
    if not 0 < fraction <= 1:
        raise ValueError("Positive activation fraction must be in (0,1]")
    result = []
    for column in np.asarray(activations).T:
        values = column[fitting_rows]
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Finite nonnegative member activations required")
        positive = values[values > 0]
        cutoff = float(np.quantile(positive, 1 - fraction)) if len(positive) >= 2 else None
        result.append({"cutoff": cutoff, "fraction": fraction, "positive_fit": len(positive),
            "ties": int((positive == cutoff).sum()) if cutoff is not None else 0,
            "status": "available" if cutoff is not None else "inactive fitting factor" if not len(positive) else "insufficient positive fitting activations", "comparison": ">"})
    return result

def member_transfer(rules, X, activations, cutoffs, patients, *, replicates=500):
    if len(rules) != 3 or np.asarray(activations).shape != (len(X), 3) or len(cutoffs) != 3:
        raise ValueError("Three members on identical patient rows required")
    if any(cutoff is None for cutoff in cutoffs):
        return {"status": "inactive member; full matrix unavailable", "matrix": None}
    masks = np.column_stack([rule.mask(X) for rule in rules])
    tails = np.asarray(activations) > np.asarray(cutoffs)
    scores = lambda rows: np.array([fidelity(tails[rows, j], masks[rows, i])["f2"] if tails[rows, j].any() else np.nan for i in range(3) for j in range(3)])
    matrix, f2 = [], scores(np.arange(len(X))).reshape(3, 3)
    for i in range(3):
        for j in range(3):
            metrics = fidelity(tails[:, j], masks[:, i])
            value = f2[i, j]
            matrix.append({"rule_member": i, "target_member": j, **metrics,
                "f2_available": float(value) if np.isfinite(value) else None,
                "change_from_source_own": float(value - f2[i, i]) if np.isfinite(value + f2[i, i]) else None,
                "change_from_target_own": float(value - f2[j, j]) if np.isfinite(value + f2[j, j]) else None,
                "relative_to_source_own": float(value / f2[i, i]) if np.isfinite(value) and f2[i, i] > 0 else None})
    overlap = lambda a, b: float((a & b).sum() / (a | b).sum()) if (a | b).any() else None
    features = [{condition.feature_name for rule in own.rules for condition in rule.conditions} for own in rules]
    conditions = [{(condition.feature_name, condition.operator, condition.threshold) for rule in own.rules for condition in rule.conditions} for own in rules]
    set_overlap = lambda a, b: len(a & b) / len(a | b) if a | b else None
    def paired_scores(rows):
        values = scores(rows).reshape(3, 3)
        metrics = [fidelity(tails[rows, j], masks[rows, i]) for i in range(3) for j in range(3)]
        return np.r_[[np.nan if item[key] is None else item[key] for key in ("precision", "recall") for item in metrics], values.ravel(), (values - np.diag(values)[:, None]).ravel()]
    return {"status": "available", "matrix": matrix, "paired_uncertainty": patient_intervals(patients, paired_scores, replicates=replicates),
        "interval_order": "9 precision, 9 recall, 9 F2, 9 F2 changes from source-own",
        "worst_own_f2": float(np.nanmin(np.diag(f2))) if np.isfinite(np.diag(f2)).any() else None,
        "worst_pair": min((row for row in matrix if row["rule_member"] != row["target_member"] and row["f2_available"] is not None), key=lambda row: row["f2_available"], default=None),
        "rule_jaccard": [[overlap(masks[:, i], masks[:, j]) for j in range(3)] for i in range(3)],
        "tail_jaccard": [[overlap(tails[:, i], tails[:, j]) for j in range(3)] for i in range(3)],
        "feature_jaccard": [[set_overlap(features[i], features[j]) for j in range(3)] for i in range(3)],
        "condition_jaccard": [[set_overlap(conditions[i], conditions[j]) for j in range(3)] for i in range(3)],
        "categories": "continuous evidence only; no calibrated categorical eligibility gate"}

def common_member_rule(own_rules, X_tune, targets_tune):
    from semantic_rules import RuleSet
    if len(own_rules) != 3 or np.asarray(targets_tune).shape != (len(X_tune), 3):
        raise ValueError("Three tune-only member targets required")
    if not np.asarray(targets_tune).any(axis=0).all():
        return {"rule": None, "status": "insufficient activation positives in at least one member"}
    choices = []
    for subset in range(1, 8):
        union = {rule.rule_id: rule for i, own in enumerate(own_rules) if subset & (1 << i) for rule in own.rules}
        candidate = RuleSet(tuple(union.values()))
        scores = [fidelity(targets_tune[:, i], candidate.mask(X_tune))["f2"] for i in range(3)]
        choices.append((min(scores), -len(union), subset, candidate, scores))
    best = max(choices, key=lambda item: item[:2])
    return {"rule": best[3], "tune_worst_f2": best[0], "tune_member_f2": best[4],
            "candidates": len(choices), "requires": "independent final evaluation; original R unchanged"}

def matched_nonfamily(group, dictionary, excluded_factors, fitting_rows):
    group, dictionary = np.asarray(group), np.asarray(dictionary)
    available, chosen = set(range(dictionary.shape[1])) - set(excluded_factors), []
    for column in group.T:
        if not available:
            raise ValueError("Insufficient non-family control factors")
        prevalence = float((column[fitting_rows] > 0).mean())
        selected = min(available, key=lambda j: (abs(float((dictionary[fitting_rows, j] > 0).mean()) - prevalence), j))
        chosen.append(selected); available.remove(selected)
    return dictionary[:, chosen], {"factors": chosen, "selection": "reference-fit prevalence, matched size; rank checked in mapping"}

def group_mapping(groups, nonfamily, patients, fit_rows, tune_rows, evaluation_rows, allowed_patients, *, alphas=(.1, 1., 10.), replicates=500):
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    if len(groups) != 2 or len(nonfamily) != 2 or any(np.asarray(group).ndim != 2 or not 1 <= group.shape[1] <= 5 for group in (*groups, *nonfamily)):
        raise ValueError("Two local groups and matched non-family controls, each size 1–5, required")
    if any(control.shape != group.shape for group, control in zip(groups, nonfamily)):
        raise ValueError("Non-family controls must match group size and patient rows")
    fit, tune = development_rows(patients, fit_rows, tune_rows, allowed_patients)
    evaluation = np.asarray(evaluation_rows, dtype=int)
    patient_partition({"fit": np.asarray(patients)[fit], "tune": np.asarray(patients)[tune], "evaluation": np.asarray(patients)[evaluation]})
    rng, output = np.random.default_rng(CFG["seed"]), []
    for direction in (0, 1):
        X, Y, control = groups[direction], groups[1 - direction], nonfamily[direction]
        sx, sy = StandardScaler().fit(X[fit]), StandardScaler().fit(Y[fit])
        x, y = sx.transform(X), sy.transform(Y)
        _, _, vt = np.linalg.svd(x[fit], full_matrices=False)
        choices = []
        for rank in range(1, min(3, int(np.linalg.matrix_rank(x[fit])), len(fit) - 1) + 1):
            basis = vt[:rank].T
            for alpha in alphas:
                model = Ridge(alpha=alpha).fit(x[fit] @ basis, y[fit])
                choices.append((float(np.mean((model.predict(x[tune] @ basis) - y[tune])**2)), rank, alpha, basis, model))
        if not choices or not len(evaluation):
            output.append({"direction": direction, "status": "insufficient support"})
            continue
        error, rank, alpha, basis, model = min(choices, key=lambda item: item[:3])
        shuffled = Ridge(alpha=alpha).fit(x[fit] @ basis, y[rng.permutation(fit)])
        sc = StandardScaler().fit(control[fit])
        c = sc.transform(control)
        _, _, cvt = np.linalg.svd(c[fit], full_matrices=False)
        if np.linalg.matrix_rank(c[fit]) < rank:
            output.append({"direction": direction, "status": "matched control lacks required local rank"})
            continue
        cbasis = cvt[:rank].T
        control_model = Ridge(alpha=alpha).fit(c[fit] @ cbasis, y[fit])
        predictions = {"mapping": model.predict(x[evaluation] @ basis), "constant": np.zeros_like(y[evaluation]),
            "shuffled": shuffled.predict(x[evaluation] @ basis), "nonfamily": control_model.predict(c[evaluation] @ cbasis)}
        def heldout_errors(rows):
            errors = [float(np.mean((prediction[rows] - y[evaluation][rows])**2)) for prediction in predictions.values()]
            return errors + [errors[0] - value for value in errors[1:]]
        output.append({"direction": direction, "status": "available", "rank": rank, "alpha": alpha,
            "tune_mse": error, "n_fit": len(fit), "n_tune": len(tune), "n_evaluation": len(evaluation),
            "scores": {name: {"standardized_mse": float(np.mean((prediction - y[evaluation])**2)),
                "raw_mse": float(np.mean((sy.inverse_transform(prediction) - Y[evaluation])**2)),
                "predicted_magnitude": float(np.linalg.norm(prediction, axis=1).mean())} for name, prediction in predictions.items()},
            "tried": [{"tune_mse": choice[0], "rank": choice[1], "alpha": choice[2]} for choice in choices],
            "frozen": {"source_scaler": sx, "target_scaler": sy, "basis": basis, "model": model},
            "paired_uncertainty": patient_intervals(np.asarray(patients)[evaluation], heldout_errors, replicates=replicates),
            "interval_order": "mapping/constant/shuffled/nonfamily MSE, then mapping-minus-each-control MSE",
            "activation_pearson": [[float(np.corrcoef(X[evaluation, i], Y[evaluation, j])[0, 1]) if np.std(X[evaluation, i]) > 0 and np.std(Y[evaluation, j]) > 0 else None for j in range(Y.shape[1])] for i in range(X.shape[1])],
            "control_requirement": "nonfamily groups chosen by reference-only prevalence/rank/size before scoring"})
    return output

def group_contribution(activations, decoder_directions, reference_scale):
    return (np.asarray(activations) @ np.asarray(decoder_directions)) * np.asarray(reference_scale)

def contribution_agreement(left, right, *, epsilon=1e-12):
    left, right = np.asarray(left), np.asarray(right)
    ln, rn = np.linalg.norm(left, axis=1), np.linalg.norm(right, axis=1)
    valid = (ln > epsilon) & (rn > epsilon)
    cosine = np.full(len(left), np.nan)
    cosine[valid] = (left[valid] * right[valid]).sum(axis=1) / (ln[valid] * rn[valid])
    return {"vector_error": np.linalg.norm(left - right, axis=1), "left_magnitude": ln,
            "right_magnitude": rn, "signed_cosine": cosine, "absolute_cosine": np.abs(cosine),
            "near_zero_rows": int((~valid).sum()), "epsilon": epsilon}

def embedding_interventions(embeddings, component, replacement=None):
    h, component = np.asarray(embeddings), np.asarray(component)
    replacement = np.zeros_like(component) if replacement is None else np.asarray(replacement)
    if h.shape != component.shape or h.shape != replacement.shape or not np.isfinite(h + component + replacement).all():
        raise ValueError("Finite aligned embedding components required")
    delta = replacement - component
    rng = np.random.default_rng(CFG["seed"])
    direction = rng.normal(size=h.shape)
    direction /= np.maximum(np.linalg.norm(direction, axis=1, keepdims=True), 1e-12)
    removal_control = direction * np.linalg.norm(component, axis=1, keepdims=True)
    substitution_control = direction * np.linalg.norm(delta, axis=1, keepdims=True)
    return {"original": h, "removed": h - component, "substituted": h + delta,
            "removal_norm_control": h + removal_control, "substitution_norm_control": h + substitution_control, "delta": delta,
            "contract": "original residual/decoder bias remain in h; decoder-group intervention, not CAV projection"}

def local_geometry(left, right, control_left, control_right, *, rank=1):
    def basis(directions):
        _, singular, vt = np.linalg.svd(np.asarray(directions), full_matrices=False)
        if not 1 <= rank <= min(3, int((singular > 1e-10).sum())):
            raise ValueError("Local rank unsupported")
        return vt[:rank].T
    if np.shape(left) != np.shape(control_left) or np.shape(right) != np.shape(control_right) or max(len(left), len(right)) > 5:
        raise ValueError("Local controls must match group size and ambient coordinates")
    overlap = lambda a, b: float(np.linalg.norm(basis(a).T @ basis(b), "fro")**2 / rank)
    denominator = np.linalg.norm(left, axis=1)[:, None] * np.linalg.norm(right, axis=1)[None, :]
    cosine = np.divide(np.asarray(left) @ np.asarray(right).T, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 1e-12)
    return {"signed_decoder_cosine": cosine, "absolute_decoder_cosine": np.abs(cosine),
            "local_subspace_overlap": overlap(left, right), "matched_control_overlap": overlap(control_left, control_right),
            "rank": rank, "scope": "local groups only; whole-dictionary full spans excluded"}


In [ ]:
def focused_correspondence_checks():
    from semantic_rules import Condition, Rule, RuleSet
    X = np.arange(12, dtype=float)[:, None]
    rules = [RuleSet((Rule(str(i), (Condition(0, "x", ">", float(i + 3)),)),)) for i in range(3)]
    activations = np.column_stack([np.maximum(X[:, 0] - i, 0) for i in range(3)])
    ids = np.array([f"patient-{i}" for i in range(12)])
    matrix = member_transfer(rules, X, activations, [3., 3., 3.], ids, replicates=4)
    assert len(matrix["matrix"]) == 9 and len(matrix["paired_uncertainty"]["intervals"]) == 36
    common = common_member_rule(rules, X, activations > 3)
    assert common["candidates"] == 7
    cutoffs = member_cutoffs(np.column_stack([np.zeros(12), activations[:, 0]]), np.arange(6), .5)
    assert cutoffs[0]["cutoff"] is None and cutoffs[1]["comparison"] == ">"
    a, D, scale = np.array([[2., 3.]]), np.array([[1., 0.], [0., 2.]]), np.array([2., 4.])
    c = group_contribution(a, D, scale)
    np.testing.assert_allclose(c, [[4., 24.]])
    h = np.array([[10., 40.]])  # Includes untouched residual, mean and decoder bias.
    changed = embedding_interventions(h, c, c)
    np.testing.assert_allclose(changed["substituted"], h)
    np.testing.assert_allclose(embedding_interventions(h, np.zeros_like(c))["removed"], h)
    np.testing.assert_allclose(embedding_interventions(h, c)["removed"] + c, h)
    unequal = embedding_interventions(h, c, 3 * c)
    np.testing.assert_allclose(np.linalg.norm(unequal["removal_norm_control"] - h, axis=1), np.linalg.norm(c, axis=1))
    np.testing.assert_allclose(np.linalg.norm(unequal["substitution_norm_control"] - h, axis=1), 2 * np.linalg.norm(c, axis=1))
    assert contribution_agreement(np.zeros_like(c), c)["near_zero_rows"] == 1
    geometry = local_geometry(D, D, D, D, rank=1)
    assert abs(geometry["local_subspace_overlap"] - 1) < 1e-12
    rng = np.random.default_rng(42)
    left = rng.normal(size=(24, 2)); right = left @ np.array([[1., .2], [.1, 1.]])
    maps = group_mapping([left, right], [rng.normal(size=left.shape), rng.normal(size=right.shape)],
        np.array([f"synthetic-{i}" for i in range(24)]), np.arange(12), np.arange(12, 18), np.arange(18, 24),
        [f"synthetic-{i}" for i in range(18)], alphas=(1.,), replicates=4)
    assert len(maps) == 2 and all(set(item["scores"]) == {"mapping", "constant", "shuffled", "nonfamily"} for item in maps)
    return {"status": "passed", "scope": "3x3/commonrule/paired uncertainty/bidirectional controls/contribution algebra/local geometry"}


In [ ]:
# Report: member transfer, common rule and shared-signal tests on final patients.
def correspondence_frames():
    pairs, families, signals = [], [], []
    for variant, run in RUNS.items():
        for reference, export in run["evaluate"].items():
            for family in export.get("correspondence", []):
                transfer, common = family["transfer"], family.get("common_rule", {})
                if transfer.get("status") != "available":
                    continue
                for cell in transfer["matrix"]:
                    if cell.get("status") != "available" or cell["rule_member"] == cell["target_member"]:
                        continue
                    i, j = cell["rule_member"], cell["target_member"]
                    pairs.append({"variant": variant, "reference": reference, "family": family["family"], "f2": cell["f2"],
                                  "target_own_f2": cell["f2"] - cell["change_from_target_own"], "change_from_source_own": cell["change_from_source_own"],
                                  "rule_jaccard": transfer["rule_jaccard"][i][j], "tail_jaccard": transfer["tail_jaccard"][i][j],
                                  "feature_jaccard": transfer["feature_jaccard"][i][j]})
                evaluation = [item["f2"] for item in common.get("evaluation", []) if item.get("status") == "available"]
                families.append({"variant": variant, "reference": reference, "family": family["family"], "worst_own_f2": transfer["worst_own_f2"],
                                 "worst_transfer_f2": transfer["worst_pair"]["f2"], "common_worst_f2": min(evaluation) if evaluation else None,
                                 "common_tune_worst_f2": common.get("tune_worst_f2")})
                for signal in family.get("shared_signal", []):
                    row = {"variant": variant, "reference": reference, "family": family["family"],
                           "signed_cosine": summary(signal["contribution"]["signed_cosine"]), "signed_cosine_control": summary(signal["contribution_control"]["signed_cosine"]),
                           "subspace_overlap": signal["geometry"]["local_subspace_overlap"], "subspace_overlap_control": signal["geometry"]["matched_control_overlap"]}
                    for mapping in signal["mapping"]:
                        if mapping.get("status") == "available":
                            signals.append({**row, "direction": mapping["direction"], **{f"mse_{name}": score["standardized_mse"] for name, score in mapping["scores"].items()}})
    return pd.DataFrame(pairs), pd.DataFrame(families), pd.DataFrame(signals)

def show_correspondence():
    pairs, families, signals = correspondence_frames()
    show(pairs.groupby(["variant", "reference"])[["f2", "target_own_f2", "change_from_source_own", "rule_jaccard", "tail_jaccard"]].median().unstack("variant").round(3),
         "transfer_median")
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
    sns.scatterplot(pairs, x="target_own_f2", y="f2", hue="variant", palette=PALETTE, hue_order=ORDER, alpha=.5, s=18, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
    axes[0].set(title="Transferred member rule vs target member's own rule (F2)", xlabel="target own F2", ylabel="transferred F2")
    sns.boxplot(pairs, x="reference", y="change_from_source_own", hue="variant", palette=PALETTE, hue_order=ORDER, fliersize=2, ax=axes[1])
    axes[1].axhline(0, color="grey", lw=1)
    axes[1].set_title("F2 change when a member's rule describes another member")
    sns.scatterplot(pairs, x="tail_jaccard", y="rule_jaccard", hue="feature_jaccard", palette="viridis", s=18, alpha=.6, ax=axes[2])
    axes[2].plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
    axes[2].set_title("Members: activation-tail vs rule-mask overlap (colour: feature Jaccard)")
    save(fig, "transfer")
    fig, axes = plt.subplots(2, 2, figsize=(15, 10.5))
    axes = axes.flat
    sns.scatterplot(families, x="worst_transfer_f2", y="common_worst_f2", hue="variant", palette=PALETTE, hue_order=ORDER, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
    axes[0].set(title="Common union rule vs worst member transfer", xlabel="worst transfer F2", ylabel="common rule worst F2")
    mse = signals.melt(id_vars=["variant", "reference"], value_vars=["mse_mapping", "mse_constant", "mse_shuffled", "mse_nonfamily"], var_name="model", value_name="mse")
    sns.boxplot(mse, x="model", y="mse", hue="variant", palette=PALETTE, hue_order=ORDER, fliersize=2, ax=axes[1])
    axes[1].set(title="Member→member activation mapping vs controls\n(standardized MSE)", xlabel="")
    for ax, (value, control, title) in zip(axes[2:], (("signed_cosine", "signed_cosine_control", "Contribution cosine:\nfamily member vs nonfamily control"),
                                                    ("subspace_overlap", "subspace_overlap_control", "Local subspace overlap:\nfamily member vs matched control"))):
        paired = signals.drop_duplicates(["variant", "family", value])
        sns.scatterplot(paired, x=control, y=value, hue="variant", palette=PALETTE, hue_order=ORDER, s=20, ax=ax, legend=False)
        low, high = np.nanmin(paired[[value, control]].values), np.nanmax(paired[[value, control]].values)
        ax.plot([low, high], [low, high], color="grey", lw=1, ls="--")
        ax.set(title=title, xlabel="control", ylabel="family member")
    fig.tight_layout()
    save(fig, "shared_signal")
    show(signals.assign(mapping_beats_controls=lambda d: d.mse_mapping < d[["mse_constant", "mse_shuffled", "mse_nonfamily"]].min(axis=1),
                        cosine_beats_control=lambda d: d.signed_cosine > d.signed_cosine_control,
                        overlap_beats_control=lambda d: d.subspace_overlap > d.subspace_overlap_control)
         .groupby("variant")[["mapping_beats_controls", "cosine_beats_control", "overlap_beats_control"]].mean().reindex(ORDER), "shared_signal_share")

if MODE == "report":
    show_correspondence()

## Why do frozen activation cohorts disappear?

Six checks separate (1) identity and empty/inactive/below-cutoff cohorts,
(2) frozen-context feature×domain input, (3) population composition and paired
patients, (4) scaling, preactivation and threshold margins, (5) reconstruction,
initialization/sample/convergence variability, and (6) observable/predictive links.
Run cheap cached checks broadly; 2011/2012 mechanism cases are post hoc. Choose
a noncollapse comparator using reference support before seeing future scores.

The nine-fit ceiling is per mechanism reference: three disjoint initializations,
three patient resamples with fixed initialization, three longer paired fits.
Longer minibatch runs preserve 100/300-epoch states; original full-batch fits
remain a separate optimization variant. A two-epoch synthetic CPU check is the
only fitting done during authoring. Domain inference and real fits require the
later pinned-source, budgeted experiment launch. None of these checks establish
clinical causality; 2015 outcomes remain excluded from primary interpretation.


In [ ]:
def frozen_activation(embeddings, weight, bias, mean, scale):
    h, w, b, mu, sigma = map(np.asarray, (embeddings, weight, bias, mean, scale))
    if h.ndim != 2 or w.ndim != 2 or h.shape[1] != w.shape[1] or (sigma <= 0).any() or not np.isfinite(sigma).all():
        raise ValueError("Aligned embedding/encoder coordinates and positive finite scales required")
    u = ((h - mu) / sigma) @ w.T + b
    if not np.isfinite(u).all():
        raise ValueError("Nonfinite frozen preactivation")
    return u, np.maximum(u, 0)

def disappearance_identity(h, weight, bias, mean, scale, factor, cutoff, cached_mask, identity, expected, *, dependency_status="ok", cutoff_atol=1e-5, cutoff_rtol=1e-4):
    if dependency_status != "ok":
        return {"check": 1, "status": "unavailable dependency", "reason": dependency_status}
    if object_identity(identity) != object_identity(expected):
        raise ValueError("Frozen object/row/domain identity mismatch")
    _, a = frozen_activation(h, weight, bias, mean, scale)
    values = a[:, factor]
    if np.asarray(cached_mask).shape != values.shape or not np.isin(cached_mask, [0, 1]).all():
        raise ValueError("Cached activation mask must match ordered rows")
    boundary = np.abs(values - cutoff) <= cutoff_atol + cutoff_rtol * abs(cutoff)
    mismatch = (values > cutoff) != np.asarray(cached_mask)
    if (mismatch & ~boundary).any():
        raise ValueError("Activation mask differs away from frozen cutoff boundary")
    status = "empty cohort" if not len(values) else "all-zero activation" if not (values > 0).any() else "positive but below cutoff" if not (values > cutoff).any() else "populated"
    return {"check": 1, "status": status, "rows": len(values), "positive": int((values > 0).sum()),
            "above_cutoff": int((values > cutoff).sum()), "boundary_discrepancies": int(mismatch.sum()), "cutoff_atol": cutoff_atol, "cutoff_rtol": cutoff_rtol}

def restore_frozen_context(population, roles, predictor, device, split_seed):
    from concept_subgroup_drift_pilot import fit_frozen_predictor, _prediction_frame, _feature_columns
    r, year = roles["reference_year"], population["year"]
    rows = (year == r) & (roles["row_role"] == "tabpfn_context") & ~roles["is_entrant"]
    window = predictor.get("context_window", "reference_only")
    first = min(predictor.get("context_years", [r]))
    if window != "reference_only":  # Window builds: earlier rows up to first death, domains from the first context year.
        rows |= (year >= first) & (year < r) & ~population["post_death"]
    rows = np.flatnonzero(rows)
    frame = _prediction_frame(population, rows, _feature_columns(population["X"].shape[1]))
    frame["y"] = population["y"][rows]
    system = fit_frozen_predictor(frame, _feature_columns(population["X"].shape[1]), device, seed=split_seed,
                                  **({} if window == "reference_only" else {"domain_origin": first}))
    if any(system[key] != predictor[key] for key in ("context_identity", "checkpoint_sha256")) or system["context_rows"] != predictor["context_rows_identity"]:
        raise ValueError("Rebuilt predictor context/checkpoint differs from the cached build")
    return system

def domain_grid(system, reference_frame, later_frame, later_year, device, encode):
    from concept_subgroup_drift_pilot import capture_frozen_predictions
    output = []
    for feature_period, frame in (("reference", reference_frame), ("later", later_frame)):
        for domain_year in (system["reference_year"], later_year):
            changed = frame.copy(); changed["year"] = domain_year
            batches, probabilities, cursor, batch = [], [], 0, CFG["inference_batch"]
            while cursor < len(changed):
                try:
                    captured = capture_frozen_predictions(system, changed.iloc[cursor:cursor + batch], device)
                except torch.cuda.OutOfMemoryError:
                    if batch <= 1:
                        raise
                    batch //= 2; torch.cuda.empty_cache(); log.warning("Domain inference OOM; batch=%s", batch)
                    continue
                batches.append(captured["embeddings"].detach().cpu().numpy())
                probabilities.append(captured["probabilities"].detach().cpu().numpy()); cursor += batch
            h = np.concatenate(batches) if batches else np.empty((0, system["classifier"].cached_models_[0].decoder_dict["standard"][0].in_features))
            output.append({"check": 2, "features": feature_period, "domain_year": domain_year,
                "embeddings": h, "activations": encode(h), "probabilities": np.concatenate(probabilities) if probabilities else np.empty((0, 2)),
                "identity": canonical_hash({"context": system["context_identity"], "features": feature_period,
                    "rows": list(zip(frame.patient_id.astype(str), frame.year.astype(int))),
                    "feature_values": frame[list(system["feature_names"])].to_numpy().tolist(), "domain": domain_year}),
                "label": "controlled domain input; original rows/clinical year unchanged"})
    return output

def composition_diagnostic(patients, years, activations, reference, later):
    patients, years, a = np.asarray(patients).astype(str), np.asarray(years), np.asarray(activations)
    if len(set(zip(patients, years))) != len(patients):
        raise ValueError("Composition requires one row per patient-year")
    old = set(patients[years == reference]); current = set(patients[years == later]); paired = sorted(old & current)
    rows = []
    for label, mask in (("all", years == later), ("returning", (years == later) & np.isin(patients, list(old))),
                        ("entrants", (years == later) & ~np.isin(patients, list(old)))):
        rows.append({"check": 3, "population": label, "patients": int(mask.sum()),
                     "mean_activation": np.mean(a[mask], axis=0).tolist() if mask.any() else None})
    index = {(patient, int(year)): i for i, (patient, year) in enumerate(zip(patients, years))}
    before = np.asarray([index[(p, reference)] for p in paired], dtype=int)
    after = np.asarray([index[(p, later)] for p in paired], dtype=int)
    rows.append({"check": 3, "population": "paired", "patients": len(paired),
        "mean_activation_change": np.mean(a[after] - a[before], axis=0).tolist() if len(paired) else None,
        "limitation": "pairing conditions on survival/observation; permanent roles must be preserved"})
    return rows, before, after

def scaling_diagnostic(before, after, weight, bias, mean, scale, factor, cutoff):
    u0, a0 = frozen_activation(before, weight, bias, mean, scale)
    u1, a1 = frozen_activation(after, weight, bias, mean, scale)
    delta = (np.asarray(after) - before) @ (np.asarray(weight) / scale).T
    np.testing.assert_allclose(u1 - u0, delta, rtol=1e-4, atol=1e-5)
    coordinate_terms = (np.asarray(after) - before) * np.asarray(weight)[factor] / scale
    return {"check": 4, "delta_u": delta, "selected_coordinate_mean": coordinate_terms.mean(axis=0),
        "selected_coordinate_mean_absolute": np.abs(coordinate_terms).mean(axis=0),
        "dominant_coordinates": np.argsort(-np.abs(coordinate_terms).mean(axis=0))[:20].tolist(),
        "inverse_scale": 1 / np.asarray(scale), "raw_encoder_normals": np.asarray(weight) / scale,
        "zero_margin_before": u0[:, factor], "zero_margin_after": u1[:, factor],
        "cutoff_margin_before": a0[:, factor] - cutoff, "cutoff_margin_after": a1[:, factor] - cutoff,
        "all_factor_active_before": (a0 > 0).sum(axis=0), "all_factor_active_after": (a1 > 0).sum(axis=0)}

def reconstruction_diagnostic(h, activations, decoder, decoder_bias, mean, scale, predict):
    reconstructed = (np.asarray(activations) @ decoder + decoder_bias) * scale + mean
    original_p, reconstructed_p = predict(h), predict(reconstructed)
    return {"check": 5, "reconstruction_mse": ((h - reconstructed)**2).mean(axis=1),
        "reference_mean_mse": ((h - mean)**2).mean(axis=1),
        "active_factors": (activations > 0).sum(axis=1), "inactive_factors": (activations == 0).sum(axis=1),
        "original_probability": original_p, "reconstruction_probability": reconstructed_p,
        "probability_change": reconstructed_p - original_p, "reconstructed": reconstructed}

def mechanism_recipe(original_seeds=(42, 135, 215)):
    return ([{"kind": "initialization", "seed": seed, "sample_seed": None, "epochs": 100} for seed in (1001, 2003, 3001)]
        + [{"kind": "patient_resample", "seed": original_seeds[0], "sample_seed": seed, "epochs": 100} for seed in (11, 29, 47)]
        + [{"kind": "longer", "seed": seed, "sample_seed": None, "epochs": 300} for seed in original_seeds])

def train_diagnostic_sae(standardized, patients, params, recipe, device, *, epochs=None):
    from sae import SAE
    if epochs is not None and MODE not in {"v0", "v1"}:
        raise ValueError("Epoch override belongs only to v0/v1 smoke checks")
    steps = recipe["epochs"] if epochs is None else epochs
    if steps < 1 or steps > 300:
        raise ValueError("Bounded training requires 1–300 epochs")
    patients = np.asarray(patients).astype(str)
    if len(set(patients)) != len(patients):
        raise ValueError("Diagnostic fit needs one reference row per patient")
    sample = np.arange(len(patients))
    if recipe["sample_seed"] is not None:
        sample = np.random.default_rng(recipe["sample_seed"]).choice(sample, len(sample), replace=True)
    set_seed(recipe["seed"], CFG["deterministic"])
    model = SAE(data_dimension=standardized.shape[1], scaling_factor=params["scaling_factor"],
                type=params["model_type"], use_decoder_bias=True, k=16, k_aux=64).to(device)
    initial = model.encoder.weight.detach().clone()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"])
    generator = torch.Generator().manual_seed(CFG["split_seed"])
    loader = torch.utils.data.DataLoader(torch.as_tensor(standardized[sample], dtype=torch.float32),
        batch_size=CFG["training_batch"], shuffle=True, generator=generator, num_workers=0, pin_memory=(device == "cuda"))
    losses, checkpoints, started = [], {}, time.monotonic()
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, steps + 1):
        total = torch.zeros((), device=device)
        model.train()
        for batch in loader:
            batch = batch.to(device, non_blocking=(device == "cuda"))
            optimizer.zero_grad(set_to_none=True); loss = model.loss(batch, params["alpha"])
            loss.backward(); optimizer.step(); total += loss.detach() * len(batch)
            if model.type == "TopK":
                with torch.no_grad():
                    model.decoder.weight.copy_(torch.nn.functional.normalize(model.decoder.weight, p=2, dim=0))
        value = float((total / len(sample)).detach().cpu())
        if not np.isfinite(value):
            raise RuntimeError("Nonfinite SAE training loss")
        losses.append(value)
        if epoch in {100, steps}:
            checkpoints[epoch] = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    movement = float(torch.nn.functional.cosine_similarity(initial.flatten(), model.encoder.weight.detach().flatten(), dim=0).cpu())
    memory = {"allocated": torch.cuda.max_memory_allocated(), "reserved": torch.cuda.max_memory_reserved()} if device == "cuda" else {"allocated": 0, "reserved": 0}
    return {"model": model.eval().to("cpu"), "recipe": recipe, "losses": losses, "checkpoints": checkpoints,
        "sample_identity": canonical_hash(patients[sample].tolist()), "rows": len(sample), "unique_patients": len(set(patients[sample])), "data_order_seed": CFG["split_seed"],
        "initial_to_trained_cosine": movement, "seconds": time.monotonic() - started, "peak_bytes": memory,
        "variant": "minibatch128; compare 100/300 within same longer fit; original full-batch baseline separate"}

def later_feature_connections(frozen_models, X_later, activations, years, losses=None, removal_probability_change=None, removal_loss_change=None):
    high = frozen_models["high_cutoff"] if frozen_models["high_cutoff"] is not None else float("inf")
    return {"check": 6, "continuous_transfer": {name: activation_errors(activations, model.predict(X_later), high) for name, model in frozen_models["models"].items()},
        "activation": np.asarray(activations), "valid_outcome": np.asarray(years) <= 2014,
        "loss": None if losses is None else np.where(np.asarray(years) <= 2014, losses, np.nan),
        "removal_probability_change": removal_probability_change,
        "removal_loss_change": None if removal_loss_change is None else np.where(np.asarray(years) <= 2014, removal_loss_change, np.nan),
        "interpretation": "descriptive links; failure does not uniquely identify mechanism"}

def mechanism_references(reference_support):
    eligible = {r: n for r, n in reference_support.items() if 2007 <= int(r) <= 2013 and int(r) not in {2011, 2012}}
    comparator = max(eligible, key=lambda r: (eligible[r], -int(r))) if eligible else None
    return {"posthoc_collapse_cases": [2011, 2012], "reference_support_comparator": comparator,
            "selection": "comparator chosen by reference support only, before later diagnostic scores"}


In [ ]:
def focused_disappearance_checks():
    from unittest.mock import patch
    import pandas as pd
    h = np.array([[1., 2.], [3., 4.]])
    w, bias, mean, scale = np.eye(2), np.array([-1., -1.]), np.zeros(2), np.array([1., 2.])
    u, a = frozen_activation(h, w, bias, mean, scale)
    np.testing.assert_allclose(u, [[0, 0], [2, 1]])
    parts = dict.fromkeys(IDENTITY_FIELDS, "synthetic")
    audit = disappearance_identity(h, w, bias, mean, scale, 0, 1., [False, True], parts, parts)
    assert audit["status"] == "populated"
    delta = scaling_diagnostic(h, h + 2, w, bias, mean, scale, 0, 1.)
    np.testing.assert_allclose(delta["delta_u"], [[2, 1], [2, 1]])
    tables, before, after = composition_diagnostic(["a", "b", "a", "c"], [2007, 2007, 2008, 2008], np.array([1, 2, 3, 4]), 2007, 2008)
    assert before.tolist() == [0] and after.tolist() == [2] and tables[-1]["patients"] == 1
    reconstructed = reconstruction_diagnostic(h, a, w, bias * 0, mean, scale, lambda value: value[:, 0] / 10)
    assert reconstructed["active_factors"].tolist() == [0, 2]
    recipes = mechanism_recipe()
    assert len(recipes) == 9 and len({item["seed"] for item in recipes[:3]}) == 3
    fit = train_diagnostic_sae(np.array([[0., 1.], [1., 0.], [1., 1.]], dtype=np.float32), ["a", "b", "c"],
        {"scaling_factor": 1.5, "model_type": "ReLU", "learning_rate": .001, "weight_decay": 0., "alpha": .1}, recipes[0], "cpu", epochs=2)
    assert len(fit["losses"]) == 2 and np.isfinite(fit["losses"]).all()
    assert mechanism_references({2007: 20, 2008: 40, 2011: 200})["reference_support_comparator"] == 2008
    frame = pd.DataFrame({"f0": [1., 2.], "f1": [2., 3.], "patient_id": ["a", "b"], "year": [2007, 2007]})
    later = frame.copy(); later["year"] = 2008
    system = {"reference_year": 2007, "context_identity": "synthetic", "feature_names": ("f0", "f1")}
    def capture(system, data, device):
        h = torch.tensor(data[["f0", "f1"]].to_numpy() + (data.year.to_numpy() - 2007)[:, None])
        return {"embeddings": h, "probabilities": torch.ones((len(data), 2)) / 2}
    with patch("concept_subgroup_drift_pilot.capture_frozen_predictions", capture):
        grid = domain_grid(system, frame, later, 2008, "cpu", lambda h: np.maximum(h, 0))
    assert len(grid) == 4 and frame.year.eq(2007).all() and later.year.eq(2008).all()
    np.testing.assert_allclose(grid[1]["embeddings"] - grid[0]["embeddings"], 1)
    assert grid[0]["identity"] != grid[1]["identity"]
    return {"status": "passed", "scope": "frozen scaling/masks/paired composition/reconstruction/bounded ninefit recipe/two-epoch CPU fit"}


In [ ]:
# Report: six disappearance checks (retrospective diagnosis in develop; check 6 on final patients in evaluate).
def disappearance_frames():
    identity, composition, scaling, reconstruction, grid, fits, transfer = [], [], [], [], [], [], []
    for variant, run in RUNS.items():
        for reference, value in run["disappearance"].items():
            reference = int(reference)
            identity += [{"variant": variant, "reference": reference, "lag": row["year"] - reference, **row} for row in value["identity"]]
            for row in value["composition"]:
                for table in row["tables"]:
                    composition.append({"variant": variant, "reference": reference, "lag": row["year"] - reference, "population": table["population"],
                                        "mean_activation": table.get("mean_activation", table.get("mean_activation_change"))})
            scaling += [{"variant": variant, "reference": reference, "lag": row["year"] - reference,
                         **{key: summary(row[key]) for key in ("cutoff_margin_before", "cutoff_margin_after", "zero_margin_before", "zero_margin_after", "delta_u_selected")}}
                        for row in value["scaling"]]
            reconstruction += [{"variant": variant, "reference": reference, "year": row["year"], "seed": row["seed"],
                                **{key: summary(row[key]) for key in ("reconstruction_mse", "reference_mean_mse", "probability_change", "active_factors")}}
                               for row in value["reconstruction"]]
            mechanism = value.get("mechanism")
            if mechanism:
                for row in mechanism["domain_grid"]:
                    grid += [{"variant": variant, "reference": reference, "domain_year": row["domain_year"], "features": row["features"], "family": family,
                              "above_cutoff": share} for family, share in row["medoid_above_cutoff"].items()]
                for row in mechanism["diagnostic_fits"]:
                    best = [candidate["pearson"] for matches in row["medoid_recovery"].values() for match in matches for candidate in match["candidates"][:1]]
                    fits.append({"variant": variant, "reference": reference, "later_year": mechanism["later_year"], "posthoc": mechanism["posthoc"],
                                 "active_fraction": row["active_fraction"], "initial_to_trained_cosine": row["initial_to_trained_cosine"],
                                 "final_loss": row["loss_first_100_last"][-1], "best_recovery_pearson": np.median(best) if best else None})
        for reference, export in run["evaluate"].items():
            for family in export["temporal"]["families"]:
                models = family.get("continuous_transfer", {}).get("continuous_transfer") or {}
                transfer += [{"variant": variant, "reference": reference, "lag": family["year"] - reference, "model": model, "subset": subset, "mae": score["mae"]}
                             for model, scores in models.items() for subset, score in scores.items() if score.get("mae") is not None]
    return {name: pd.DataFrame(rows) for name, rows in zip(("identity", "composition", "scaling", "reconstruction", "grid", "fits", "transfer"),
                                                           (identity, composition, scaling, reconstruction, grid, fits, transfer))}

def show_disappearance():
    frames = disappearance_frames()
    identity = frames["identity"]
    shares = identity.groupby(["variant", "lag"]).status.value_counts(normalize=True).rename("share").reset_index()
    grid = sns.catplot(shares, x="lag", y="share", hue="status", col="variant", col_order=ORDER, kind="bar", height=3.6, aspect=1.4, palette="Set2")
    grid.figure.suptitle("Check 1 — frozen member cohorts by years after reference", y=1.04)
    save(grid.figure, "disappearance_identity")
    show(identity[identity.variant == ORDER[0]].pivot_table(index="reference", columns="year", values="above_cutoff", aggfunc="median").round(1),
         f"disappearance_above_cutoff_{ORDER[0]}")
    fig, axes = plt.subplots(1, 3, figsize=(20, 4.4))
    composition = frames["composition"]
    sns.lineplot(composition[composition.variant == ORDER[0]], x="lag", y="mean_activation", hue="population", marker="o", estimator="median", ax=axes[0])
    axes[0].axhline(0, color="grey", lw=1)
    axes[0].set_title(f"Check 3 — medoid activation by population ({ORDER[0]}; paired = change)")
    scaling = frames["scaling"].melt(id_vars=["variant", "lag"], value_vars=["cutoff_margin_before", "cutoff_margin_after", "zero_margin_before", "zero_margin_after"],
                                     var_name="margin", value_name="value")
    sns.lineplot(scaling[scaling.variant == ORDER[0]], x="lag", y="value", hue="margin", estimator="median", marker="o", ax=axes[1])
    axes[1].axhline(0, color="grey", lw=1)
    axes[1].set_title("Check 4 — preactivation margins to cutoff/zero (median, paired)")
    reconstruction = frames["reconstruction"].assign(relative_mse=lambda d: d.reconstruction_mse / d.reference_mean_mse)
    sns.boxplot(reconstruction, x="reference", y="relative_mse", hue="variant", palette=PALETTE, hue_order=ORDER, fliersize=2, ax=axes[2])
    axes[2].set(yscale="log", title="Check 5 — reconstruction MSE / reference-mean MSE")
    save(fig, "disappearance_mechanisms")
    if not frames["grid"].empty:
        domain = frames["grid"][frames["grid"].variant == ORDER[0]]
        references = sorted(domain.reference.unique())
        fig, axes = plt.subplots(1, len(references), figsize=(7.5 * len(references), 4.8), squeeze=False)
        for ax, reference in zip(axes[0], references):
            heat = domain[domain.reference == reference].pivot_table(index="family", columns=["features", "domain_year"], values="above_cutoff")
            heat.index = [name.split("/", 2)[2] for name in heat.index]
            heatmap(heat, ax, cmap="rocket_r", vmin=0, cbar=False)
            ax.tick_params(axis="y", rotation=0)
            ax.set(title=f"Check 2 — share above frozen cutoff, reference {reference}\n(input features × TabPFN domain)", xlabel="features · domain year", ylabel="")
        fig.tight_layout()
        save(fig, "disappearance_domain_grid")
        show(frames["fits"], "disappearance_diagnostic_fits", index=False)
    transfer = frames["transfer"]
    if not transfer.empty:
        grid = sns.relplot(transfer[transfer.subset.isin(["full", "positive"])], x="lag", y="mae", hue="model", style="subset", col="variant", col_order=ORDER,
                           kind="line", estimator="median", errorbar=None, marker="o", height=3.6, aspect=1.3)
        grid.figure.suptitle("Check 6 — continuous activation models carried forward (median MAE, final patients)", y=1.04)
        save(grid.figure, "disappearance_continuous_transfer")

if MODE == "report":
    show_disappearance()

## The two temporal ideas

Load the original frozen rule/target/CAV references and retain their separate
Idea 1/Idea 2 eligibility. The originally eligible families are distinct from all
frozen rules and the broader full-dictionary collapse audit. Members are recorded
before family summaries. Frozen A, frozen R and original cutoff conventions stay
fixed; new yearly descriptions never replace them.

Idea 1 follows activation prevalence/magnitude, rule fidelity, observed features,
coactivation and frozen continuous prediction. Idea 2 follows retention within R,
full frozen/yearly dictionary profiles and same-patient correspondence. Match
yearly dictionaries using available same-year fitting patients; score descriptions
on independent evaluation patients. Ambiguous, unmatched and split/merge candidates
remain visible. Missing 2014/2015 units retain their failure status. References
2007–2013 are primary; representations extend through 2015 without primary
mortality claims. Reference-only recipes must be frozen before final scoring.


In [ ]:
def activation_profile(codes, mask):
    selected = np.asarray(codes)[np.asarray(mask, dtype=bool)]
    return {"rows": len(selected), "positive_prevalence": (selected > 0).mean(axis=0).tolist() if len(selected) else None,
            "mean_magnitude": selected.mean(axis=0).tolist() if len(selected) else None,
            "status": "available" if len(selected) else "empty subgroup"}

def yearly_correspondence(reference_codes, yearly_codes, factors, matching_rows, reporting_mask, *, minimum_similarity=None):
    a, b = np.asarray(reference_codes), np.asarray(yearly_codes)
    if a.shape[0] != b.shape[0] or np.asarray(reporting_mask).shape != (len(a),):
        raise ValueError("Yearly matching requires identical ordered patient rows")
    if len(matching_rows) < 3:
        return {"status": "insufficient fitting patients", "members": []}
    x, y = a[matching_rows], b[matching_rows]
    xc, yc = x - x.mean(axis=0), y - y.mean(axis=0)
    denominator = np.linalg.norm(xc, axis=0)[:, None] * np.linalg.norm(yc, axis=0)[None, :]
    correlation = np.divide(xc.T @ yc, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 1e-12)
    records = []
    for factor in factors:
        finite = np.flatnonzero(np.isfinite(correlation[factor]))
        ranked = sorted(finite, key=lambda j: (-correlation[factor, j], j))
        candidates = [j for j in ranked if correlation[factor, j] >= minimum_similarity] if minimum_similarity is not None else ranked[:3]
        records.append({"reference_factor": int(factor), "status": "unmatched" if not candidates else "candidate split/ambiguity" if len(candidates) > 1 else "candidate correspondence",
            "candidates": [{"factor": int(j), "pearson": float(correlation[factor, j]),
                "profile_inside_R": activation_profile(b[:, [j]], reporting_mask)} for j in candidates],
            "criterion": minimum_similarity, "qualification": "top candidates descriptive; threshold requires frozen reference calibration"})
    counts = {j: sum(any(item["factor"] == j for item in record["candidates"]) for record in records) for j in range(b.shape[1])}
    return {"status": "available", "members": records, "possible_merges": [j for j, n in counts.items() if n > 1],
            "yearly_full_profile": activation_profile(b, reporting_mask), "matching_patients": len(matching_rows)}

def temporal_inputs(population, cache, config, reference, device):
    from concept_subgroup_drift_pilot import reference_year_roles, load_frozen_references, load_reference_sae, encode_with_reference_sae
    required = [f"P/{reference}", *[f"D/{reference}/{stage}" for stage in ("families", "rules", "cavs_selection", "reference_performance")], *[f"S/{seed}/{reference}" for seed in config["sae_seeds"]]]
    if not all(cache.is_complete(key) for key in required):
        return {"status": "unavailable dependency", "units": cache.completeness_table(required)}
    predictor, roles = cache.load(f"P/{reference}"), reference_year_roles(population, reference, config["split_seed"])
    rows = np.asarray(predictor["row_index"])
    if not np.array_equal(rows, np.flatnonzero(population["year"] >= reference)) or predictor["roles_identity"] != roles["identity"] or predictor["population_identity"] != population["fingerprint"]["population_identity"]:
        raise ValueError("Temporal population, role or ordered-row identity mismatch")
    references = load_frozen_references(cache, reference)
    selection, performance, families = [cache.load(f"D/{reference}/{stage}") for stage in ("cavs_selection", "reference_performance", "families")]
    if any(performance["references"].get(key) != value["identity"] for key, value in references.items()):
        raise ValueError("Reference performance object identity mismatch")
    selected = {key: value for key, value in references.items() if any(value["eligible"].values())}
    codes, dictionary_ids = {}, {}
    for seed in config["sae_seeds"]:
        dictionary = load_reference_sae(cache, reference, seed, device)
        codes[seed] = encode_with_reference_sae(dictionary, predictor["embeddings"], device, CFG["encoding_batch"])
        dictionary_ids[seed] = {key: dictionary[key] for key in ("scaler_identity", "sae_identity", "unit_identity")}
        dictionary["run"]["model"].to("cpu")
    return {"status": "available", "predictor": predictor, "roles": roles, "rows": rows,
        "references": selected, "families": {item["id"]: item for item in families["families"]},
        "selection": selection, "performance": performance, "codes": codes, "dictionary_ids": dictionary_ids,
        "selection_count": {"frozen_rules": len(references), "eligible_any_idea": len(selected)}}

def yearly_description(X, activations, cutoff, patients, fit_pool, evaluation, names, recipe, year, family_id):
    from concept_subgroup_drift_pilot import rule_inner_split
    X, names = explanation_features(X, names, **recipe.get("transform", {"kind": "counts"}))
    if len(fit_pool) < 4 or not len(evaluation):
        return {"status": "insufficient independent yearly fitting/evaluation patients"}
    split_seed = int(canonical_hash([CFG["split_seed"], year, family_id]).split(":")[1][:8], 16)
    fit_ids, tune_ids = rule_inner_split(patients[fit_pool], (activations[fit_pool] > cutoff).astype(float), split_seed)
    fit = fit_pool[np.isin(patients[fit_pool], fit_ids)]; tune = fit_pool[np.isin(patients[fit_pool], tune_ids)]
    if not len(fit) or not len(tune) or not (activations[fit] > cutoff).any() or not (activations[tune] > cutoff).any():
        return {"status": "insufficient positive yearly fit/tune support"}
    descriptions = fit_member_explanation(X, names, activations, cutoff, patients, fit, tune, patients[fit_pool], recipe["backend"], recipe["selection"], candidate_policies(recipe["policy"]))
    chosen = descriptions[0]
    return {"status": "available", "rule": chosen["rule_dict"], "independent_fidelity": fidelity(activations[evaluation] > cutoff, chosen["rule"].mask(X[evaluation])),
            "fit": chosen["fit_identity"], "tune": chosen["tune_identity"], "primary_A": "unchanged frozen reference cutoff"}

def follow_both_ideas(population, cache, config, reference, device, *, rule_recipe=None, continuous_models=None, minimum_similarity=None):
    from concept_subgroup_drift_pilot import load_yearly_sae, encode_with_reference_sae
    if reference not in range(2007, 2016):
        return {"status": "reference outside 2007-2015", "units": cache.completeness_table([f"P/{reference}"])}
    state = temporal_inputs(population, cache, config, reference, device)
    if state["status"] != "available":
        return state
    rows, roles, codes = state["rows"], state["roles"], state["codes"]
    X, years, patients = population["X"][rows], population["year"][rows], population["patient_id"][rows]
    evaluation_role, fitting_role = roles["row_role"][rows] == "evaluation", np.isin(roles["row_role"][rows], ["rule_discovery", "fitting"])
    reference_fit = np.flatnonzero((years == reference) & (roles["row_role"][rows] == "rule_discovery"))
    members, families, broad = [], [], []
    for year in range(reference, 2016):
        at = years == year; evaluation = np.flatnonzero(at & evaluation_role); fitting = np.flatnonzero(at & fitting_role)
        available, yearly = np.flatnonzero(at), {}
        for s in config["sae_seeds"] if year > reference and any(item["eligible"]["idea2"] for item in state["references"].values()) else []:
            key = f"D/{reference}/idea2_yearly_sae/{s}/{year}"
            if not cache.is_complete(key):
                yearly[s] = {"status": "unavailable", "unit": cache.completeness_table([key])}; continue
            unit = cache.load(key)
            if unit.get("status") != "ok":
                yearly[s] = {"status": "unavailable", "reason": unit.get("reason")}; continue
            dictionary = load_yearly_sae(cache, reference, s, year, device)
            if dictionary["scaler_identity"] != state["dictionary_ids"][s]["scaler_identity"]:
                raise ValueError("Yearly dictionary scaler identity changed")
            yearly_codes = encode_with_reference_sae(dictionary, state["predictor"]["embeddings"][available], device, CFG["encoding_batch"])
            factors = sorted({int(k) for family_id in state["references"] for ss, k in state["families"][family_id]["members"] if int(ss) == s})
            matches = yearly_correspondence(codes[s][available], yearly_codes, factors, np.flatnonzero(fitting_role[available]), np.isin(available, evaluation), minimum_similarity=minimum_similarity)
            yearly[s] = {"status": "available", "codes": yearly_codes, "matches": matches, "dictionary_identity": dictionary["unit_identity"]}
            dictionary["run"]["model"].to("cpu")
        for seed, full_codes in codes.items():
            broad.append({"reference": reference, "year": year, "seed": seed, "all_factors": activation_profile(full_codes, at & evaluation_role)})
        for family_id, frozen in state["references"].items():
            family = state["families"][family_id]; seed, factor = map(int, frozen["medoid"])
            values, rule = codes[seed][:, factor], frozen["rule"]
            A, R = values[evaluation] > frozen["cutoff"], rule.mask(X[evaluation])
            base_rows = np.flatnonzero((years == reference) & evaluation_role)
            base_R = rule.mask(X[base_rows]); base_retention = float((values[base_rows][base_R] > frozen["cutoff"]).mean()) if base_R.any() else None
            retention = float(A[R].mean()) if R.any() else None
            in_A = at & evaluation_role & np.isin(np.arange(len(rows)), evaluation[A])
            in_R = at & evaluation_role & np.isin(np.arange(len(rows)), evaluation[R])
            positive = values[evaluation][values[evaluation] > 0]
            record = {"reference": reference, "year": year, "family": family_id, "reference_identity": frozen["identity"],
                "eligible": frozen["eligible"], "rows": len(evaluation), "status": "available" if len(evaluation) else "empty evaluation cohort",
                "A_rows": int(A.sum()), "R_rows": int(R.sum()), "A_prevalence": float(A.mean()) if len(A) else None,
                "A_positive_prevalence": float((values[evaluation] > 0).mean()) if len(A) else None,
                "A_magnitude": float(values[evaluation].mean()) if len(A) else None,
                "A_high_mean": float(values[evaluation][A].mean()) if A.any() else None,
                "A_positive_mean": float(positive.mean()) if len(positive) else None,
                "idea_status": {idea: "eligible" if eligible else "not eligible at R1 selection" for idea, eligible in frozen["eligible"].items()},
                "observable_A_mean": X[evaluation][A].mean(axis=0).tolist() if A.any() else None,
                "observable_R_mean": X[evaluation][R].mean(axis=0).tolist() if R.any() else None,
                "frozen_rule_fidelity": fidelity(A, R) if len(A) else None, "retention": retention,
                "reference_retention": base_retention, "retention_ratio": retention / base_retention if retention is not None and base_retention else None,
                "frozen_profiles_inside_R": {str(s): activation_profile(c, in_R) for s, c in codes.items()},
                "coactivation_inside_A": {str(s): activation_profile(c, in_A) for s, c in codes.items()}}
            for member_seed, member_factor in family["members"]:
                member_seed, member_factor = int(member_seed), int(member_factor)
                vector = codes[member_seed][:, member_factor]
                target = member_cutoffs(vector[:, None], reference_fit, frozen["activation_target"])[0]
                cutoff = frozen["cutoff"] if (member_seed, member_factor) == (seed, factor) else target["cutoff"]
                members.append({"reference": reference, "year": year, "family": family_id, "seed": member_seed, "factor": member_factor,
                    "cutoff": cutoff, "positive_fraction": frozen["activation_target"], "cutoff_status": target["status"],
                    "inside_A": activation_profile(vector[:, None], in_A),
                    "inside_R": activation_profile(vector[:, None], in_R),
                    "retention": float((vector[evaluation][R] > cutoff).mean()) if R.any() and cutoff is not None else None})
            record["yearly_description"] = yearly_description(X, values, frozen["cutoff"], patients, fitting, evaluation, list(population["feature_names"]), rule_recipe, year, family_id) if rule_recipe and frozen["eligible"]["idea1"] else {"status": "not run: no frozen recipe or ineligible Idea 1"}
            bundle = (continuous_models or {}).get(family_id)
            if bundle and len(evaluation):
                features, _ = explanation_features(X[evaluation], list(population["feature_names"]), **bundle["transform"])
                record["continuous_transfer"] = later_feature_connections(bundle["models"], features, values[evaluation], years[evaluation])
            record["yearly_representations"] = []
            for s in config["sae_seeds"] if frozen["eligible"]["idea2"] and year > reference else []:
                entry = yearly[s]
                if entry["status"] != "available":
                    record["yearly_representations"].append({"seed": s, **entry}); continue
                rmask = np.isin(available, evaluation[R]); own_factors = {int(k) for ss, k in family["members"] if int(ss) == s}
                family_matches = []
                for member in entry["matches"]["members"]:
                    if member["reference_factor"] in own_factors:
                        candidates = [{**candidate, "profile_inside_R": activation_profile(entry["codes"][:, [candidate["factor"]]], rmask)} for candidate in member["candidates"]]
                        family_matches.append({**member, "candidates": candidates})
                matches = {**entry["matches"], "members": family_matches, "yearly_full_profile": activation_profile(entry["codes"], rmask)}
                record["yearly_representations"].append({"seed": s, "dictionary_identity": entry["dictionary_identity"], **matches})
            families.append(record)
    return {"status": "available", "members": members, "families": families, "all_dictionary_factors": broad,
            "selection_count": state["selection_count"], "state": state, "interpretation": "patient-separated exploratory R1 continuation; missing years never bridged"}


In [ ]:
def focused_temporal_checks():
    from unittest.mock import patch
    from semantic_rules import Condition, Rule, RuleSet
    assert fidelity([], [])["f2"] is None and fidelity([False], [False])["jaccard"] is None
    assert fidelity([False], [True])["lift"] is None and fidelity([True], [False])["f2"] == 0
    profile = activation_profile(np.array([[0., 1.], [2., 0.]]), [True, False])
    assert profile["rows"] == 1 and profile["positive_prevalence"] == [0., 1.]
    assert activation_profile(np.zeros((2, 2)), [False, False])["mean_magnitude"] is None
    x = np.column_stack([np.arange(8), np.ones(8), np.arange(8)])
    y = np.column_stack([np.arange(8), np.arange(8), -np.arange(8)])
    result = yearly_correspondence(x, y, [0, 1, 2], np.arange(5), np.array([False] * 5 + [True] * 3), minimum_similarity=.8)
    assert len(result["members"][0]["candidates"]) == 2
    assert result["members"][1]["status"] == "unmatched"
    assert result["possible_merges"] == [0, 1]
    assert result["yearly_full_profile"]["rows"] == 3
    assert yearly_correspondence(x, y, [0], [0, 1], np.ones(8, dtype=bool))["status"] == "insufficient fitting patients"
    rule = RuleSet((Rule("synthetic", (Condition(0, "x", ">", 1.),)),))
    population = {"X": np.arange(10, dtype=float)[:, None], "year": np.array([2007] * 5 + [2008] * 5),
        "patient_id": np.array(["a", "b", "c", "d", "e"] * 2), "feature_names": ["x"]}
    reference = {"medoid": [42, 0], "rule": rule, "cutoff": 3.5, "identity": "synthetic", "activation_target": .5, "eligible": {"idea1": False, "idea2": True}}
    state = {"status": "available", "rows": np.arange(10), "roles": {"row_role": np.array((["rule_discovery"] * 3 + ["evaluation"] * 2) * 2)},
        "predictor": {"embeddings": np.zeros((10, 2))}, "dictionary_ids": {42: {"scaler_identity": "same"}},
        "codes": {42: np.arange(10, dtype=float)[:, None]}, "references": {"family": reference},
        "families": {"family": {"members": [[42, 0]]}}, "selection_count": {"frozen_rules": 2, "eligible_any_idea": 1}}
    class Cache:
        def is_complete(self, key): return key.endswith("/2008")
        def load(self, key): return {"status": "ok"}
        def completeness_table(self, keys): return [{"unit": key, "status": "not run"} for key in keys]
    dictionary = {"scaler_identity": "same", "unit_identity": "yearly", "run": {"model": torch.nn.Identity()}}
    with patch.dict(follow_both_ideas.__globals__, {"temporal_inputs": lambda *args: state}), patch("concept_subgroup_drift_pilot.load_yearly_sae", lambda *args: dictionary), patch("concept_subgroup_drift_pilot.encode_with_reference_sae", lambda *args: np.column_stack([np.arange(5), np.arange(5), np.ones(5)])):
        result = follow_both_ideas(population, Cache(), {"sae_seeds": [42]}, 2007, "cpu", minimum_similarity=.8)
    assert result["selection_count"]["eligible_any_idea"] == 1 and len(result["members"]) == 9
    first = result["families"][0]
    assert first["A_rows"] == 1 and first["R_rows"] == 2 and first["idea_status"]["idea1"].startswith("not eligible")
    assert result["families"][1]["yearly_representations"][0]["dictionary_identity"] == "yearly"
    return {"status": "passed", "scope": "frozen profiles/empty support/same-patient yearly matching/split ambiguity/constant unmatched"}


In [ ]:
# Report: Idea 1 (fixed factor) and Idea 2 (fixed rule) trajectories on final patients; missing years stay gaps.
def temporal_frame():
    rows, dictionary = [], []
    for variant, run in RUNS.items():
        for reference, export in run["evaluate"].items():
            for family in export["temporal"]["families"]:
                frozen, yearly = family.get("frozen_rule_fidelity") or {}, (family.get("yearly_description") or {}).get("independent_fidelity") or {}
                rows.append({"variant": variant, "reference": reference, "family": family["family"], "year": family["year"], "lag": family["year"] - reference,
                             "A_prevalence": family.get("A_prevalence"), "A_positive_prevalence": family.get("A_positive_prevalence"),
                             "A_magnitude": family.get("A_magnitude"), "retention": family.get("retention"), "retention_ratio": family.get("retention_ratio"),
                             "frozen_rule_f2": frozen.get("f2"), "frozen_rule_coverage": frozen.get("coverage"), "yearly_rule_f2": yearly.get("f2"),
                             "idea1": family["idea_status"]["idea1"], "idea2": family["idea_status"]["idea2"]})
            for row in export["temporal"]["all_dictionary_factors"]:
                prevalence = np.asarray(row["all_factors"].get("positive_prevalence") or [], dtype=float)
                if prevalence.size:
                    dictionary.append({"variant": variant, "reference": reference, "seed": row["seed"], "year": row["year"], "lag": row["year"] - reference,
                                       "alive_share": float((prevalence > 0).mean()), "median_prevalence": float(np.median(prevalence)), "prevalence": prevalence})
    return pd.DataFrame(rows), pd.DataFrame(dictionary)

def feature_names():
    names = {}
    for run in RUNS.values():
        for members in run["development"].values():
            for member in members:
                for result in member.get("results", []):
                    if result["transform"]["kind"] == "counts":
                        for item in result["policies"]:
                            for rule in item["rule_dict"]["rules"]:
                                names.update({condition["feature_index"]: condition["feature_name"] for condition in rule["conditions"]})
    return names

def show_temporal():
    frame, dictionary = temporal_frame()
    references = sorted(frame.reference.unique())
    for key, title in (("A_prevalence", "Idea 1 — frozen factor cohort prevalence"), ("frozen_rule_f2", "Idea 2 — frozen rule fidelity to frozen target (F2)")):
        columns = 5
        fig, axes = plt.subplots(int(np.ceil(len(references) / columns)), columns, figsize=(4.2 * columns, 3.2 * int(np.ceil(len(references) / columns))),
                                 sharex=True, sharey=True, squeeze=False)
        for ax, reference in zip(axes.flat, references):
            subset = frame[(frame.reference == reference) & (frame.variant == ORDER[0])]
            sns.lineplot(subset, x="year", y=key, units="family", estimator=None, color=PALETTE[ORDER[0]], alpha=.35, lw=1, ax=ax)
            for variant in ORDER:
                median = frame[(frame.reference == reference) & (frame.variant == variant)].groupby("year")[key].median()
                ax.plot(median.index, median.values, color=PALETTE[variant], lw=2.4, marker="o", ms=3, label=variant)
            ax.set(title=f"reference {reference}", xticks=range(2007, 2016, 2), xlim=(2006.6, 2015.6))
            ax.axvspan(LAST_OUTCOME + .5, 2015.5, color="grey", alpha=.08)
        for ax in axes.flat[len(references):]:
            ax.axis("off")
        axes.flat[0].legend(fontsize=7)
        fig.suptitle(f"{title}: thin = {ORDER[0]} families, thick = variant medians", y=1.01)
        fig.tight_layout()
        save(fig, f"temporal_{key}")
    long = frame.melt(id_vars=["variant", "lag"], value_vars=["A_prevalence", "retention", "frozen_rule_f2", "yearly_rule_f2"], var_name="measure")
    grid = sns.relplot(long, x="lag", y="value", hue="variant", col="measure", col_wrap=4, kind="line", palette=PALETTE, hue_order=ORDER, estimator="median",
                       errorbar=("pi", 50), marker="o", height=3.4, aspect=1.15, facet_kws={"sharey": False})
    grid.figure.suptitle("Trajectories by years after reference (median, IQR across families)", y=1.04)
    save(grid.figure, "temporal_by_lag")
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.6))
    sns.scatterplot(frame[frame.lag > 0], x="frozen_rule_f2", y="yearly_rule_f2", hue="variant", palette=PALETTE, hue_order=ORDER, s=14, alpha=.5, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], color="grey", ls="--", lw=1)
    axes[0].set_title("Later years: yearly refit rule vs frozen rule (F2)")
    sns.lineplot(dictionary, x="lag", y="alive_share", hue="variant", palette=PALETTE, hue_order=ORDER, estimator="median", marker="o", ax=axes[1])
    axes[1].set(title="Full dictionaries: share of factors ever active", ylim=(0, 1.02))
    sns.lineplot(dictionary, x="lag", y="median_prevalence", hue="variant", palette=PALETTE, hue_order=ORDER, estimator="median", marker="o", ax=axes[2], legend=False)
    axes[2].set_title("Full dictionaries: median factor prevalence")
    save(fig, "temporal_dictionary")
    example = dictionary[(dictionary.variant == ORDER[0]) & (dictionary.seed == dictionary.seed.min())]
    first = example.reference.min()
    heat = np.vstack(example[example.reference == first].sort_values("year").prevalence.values)
    order = np.argsort(-heat[0])
    fig, ax = plt.subplots(figsize=(14, 3.2))
    sns.heatmap(heat[:, order], cmap="mako", ax=ax, yticklabels=sorted(example[example.reference == first].year), xticklabels=False)
    ax.set(title=f"Reference {first} dictionary (seed {example.seed.min()}): factor positive prevalence per year, factors sorted by reference year",
           xlabel="factor", ylabel="year")
    save(fig, "temporal_dictionary_heatmap")
    names = feature_names()
    for variant in ORDER[:1]:
        families = [family for export in RUNS[variant]["evaluate"].values() for family in export["temporal"]["families"]]
        first = min(family["reference"] for family in families)
        years = pd.Series([family["family"] for family in families if family["reference"] == first and family.get("observable_A_mean")]).value_counts()
        chosen = sorted(years[years == years.max()].index)[:3]
        fig, axes = plt.subplots(1, len(chosen), figsize=(7.5 * len(chosen), 5.4), squeeze=False)
        for ax, name in zip(axes[0], chosen):
            series = sorted((family for family in families if family["family"] == name and family.get("observable_A_mean")), key=lambda family: family["year"])
            profile = np.array([family["observable_A_mean"] for family in series], dtype=float)
            top = np.argsort(-profile[0])[:15]
            heatmap(pd.DataFrame(profile[:, top].T, index=[names.get(int(i), f"feature {i}")[:28] for i in top], columns=[family["year"] for family in series]),
                    ax, fmt=".1f", size=6, cmap="YlOrBr", cbar=False)
            ax.set_title(f"Idea 1 meaning: mean observables inside A\n{name}", fontsize=9)
        fig.tight_layout()
        save(fig, "temporal_observable_profiles")
    show(frame.groupby(["variant", "lag"])[["A_prevalence", "retention", "frozen_rule_f2", "yearly_rule_f2"]].median().unstack("variant").round(3), "temporal_by_lag")

if MODE == "report":
    show_temporal()

## Performance and influence throughout time

For A and R, record annual Brier loss, skill against the independently fitted
frozen reference baseline, calibration and classification support. Compare TCAV
with signed/absolute sensitivity for each original CAV seed. Observed constancy
is reported without inventing structural certainty or refitting directions.

Concurrent levels and consecutive-year changes relate performance to influence,
activation prevalence/magnitude and observable/internal alignment. Patient-cluster
uncertainty respects repeated patients; between-concept summaries remain descriptive
because families overlap. These associations do not establish causation. All 2015
outcome scores are unavailable while representation/influence remains interpretable.

Budgeted interventions replay the frozen head, sample all or up to 256 patients
by identity hash, and preserve residual/bias with separate norm controls. Positive
removal loss change means removal hurt the specified prediction. Private patient
arrays remain in memory/on Fiji and must never enter aggregate exports. This
track reuses original R1 baseline identities; changed rules require new fitting
baselines before they can be scored as a redesigned experiment.


In [ ]:
def classification(y, p, cut):
    predicted = p >= cut; tp = int((predicted & (y == 1)).sum()); tn = int((~predicted & (y == 0)).sum())
    fp, fn = int(predicted.sum()) - tp, int((y == 1).sum()) - tp
    return {"threshold": float(cut), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "accuracy": (tp + tn) / len(y),
        "sensitivity": tp / (tp + fn) if tp + fn else None, "specificity": tn / (tn + fp) if tn + fp else None,
        "precision": tp / (tp + fp) if tp + fp else None, "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
        "predicted_positive_rate": float(predicted.mean())}

def frozen_threshold(labels, probabilities):
    """Max Death-F1 over distinct probability boundaries (death if p >= threshold); ties: higher precision, then higher threshold."""
    y, p = np.asarray(labels, dtype=int), np.asarray(probabilities, dtype=float)
    if not ((y == 1).any() and (y == 0).any()):
        return {"status": "unavailable: needs deaths and survivors", "threshold": None, "rows": len(y)}
    order = np.argsort(-p, kind="mergesort"); scores = p[order]
    last = np.r_[scores[1:] != scores[:-1], True]
    tp, selected = np.cumsum(y[order])[last], np.arange(1, len(y) + 1)[last]
    f1, precision = 2 * tp / (selected + y.sum()), tp / selected
    best = max(range(len(tp)), key=lambda i: (f1[i], precision[i], scores[last][i]))
    return {"status": "available", "threshold": float(scores[last][best]), "death_f1": float(f1[best]), "precision": float(precision[best]),
        "recall": float(tp[best] / y.sum()), "rows": len(y), "deaths": int(y.sum()), "candidates": int(last.sum())}

def mortality_metrics(labels, probabilities, q, year, threshold=None):
    p = np.asarray(probabilities, dtype=float)
    if year > 2014:
        return {"status": "invalid mortality recording", "rows": len(p), "brier": None, "skill": None}
    y = np.asarray(labels)
    if y.shape != p.shape or not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Aligned binary mortality and finite probabilities required")
    if not len(y):
        return {"status": "empty subgroup", "rows": 0, "brier": None, "skill": None}
    loss = float(np.mean((p - y)**2)); baseline = float(np.mean((q - y)**2)) if q is not None else None
    bins = np.minimum((p * 5).astype(int), 4)
    return {"status": "available", "rows": len(y), "deaths": int(y.sum()), "survivors": int((y == 0).sum()),
        "brier": loss, "baseline_brier": baseline, "frozen_fitting_q": q,
        "skill": 1 - loss / baseline if baseline is not None and baseline > 0 else None,
        "skill_status": "available" if baseline is not None and baseline > 0 else "missing baseline or zero baseline loss",
        "mean_risk": float(p.mean()), "mortality": float(y.mean()), "calibration_gap": float(p.mean() - y.mean()),
        **classification(y, p, .5), "at_frozen_threshold": classification(y, p, threshold) if threshold is not None else None,
        "calibration_bins": [{"bin": i, "n": int((bins == i).sum()), "risk": float(p[bins == i].mean()) if (bins == i).any() else None,
                              "mortality": float(y[bins == i].mean()) if (bins == i).any() else None} for i in range(5)]}

def predictor_performance(population, state, reference, threshold):
    """Predictor-level annual scores on all evaluation rows at 0.5, the frozen threshold and a per-year oracle (diagnostic only)."""
    from sklearn.metrics import average_precision_score, roc_auc_score
    rows, predictor, role = state["rows"], state["predictor"], state["roles"]["row_role"][state["rows"]]
    cut = (threshold or {}).get("threshold")
    output = []
    for year in range(reference, 2016):
        at = np.flatnonzero((population["year"][rows] == year) & (role == "evaluation"))
        if year > CFG["primary_outcome_last_year"]:
            output.append({"reference": reference, "year": year, "rows": len(at), "status": "invalid mortality recording"})
            continue
        y, p = population["y"][rows[at]], predictor["probabilities"][at, 1]
        both = len(np.unique(y)) == 2
        output.append({"reference": reference, "year": year, "frozen_threshold": cut, "metrics": mortality_metrics(y, p, None, year, cut),
            "average_precision": float(average_precision_score(y, p)) if both else None, "auroc": float(roc_auc_score(y, p)) if both else None,
            "oracle_diagnostic": frozen_threshold(y, p)})
    return output

def correlation_summary(x, y):
    pairs = [(a, b) for a, b in zip(x, y) if a is not None and b is not None and np.isfinite(a) and np.isfinite(b)]
    a, b = np.array(pairs, dtype=float).T if pairs else (np.array([]), np.array([]))
    valid = len(a) >= 3 and np.ptp(a) > 1e-12 and np.ptp(b) > 1e-12
    return {"n": len(a), "pearson": float(np.corrcoef(a, b)[0, 1]) if valid else None,
            "status": "descriptive" if valid else "insufficient support or constant series"}

def sensitivity_summary(sensitivity, patients, labels, probabilities, year, *, replicates=500):
    s = np.asarray(sensitivity)
    if not len(s):
        return {"status": "empty subgroup", "tcav": None, "signed": None, "absolute": None}
    result = {"status": "available", "tcav": float((s > 0).mean()), "signed": float(s.mean()), "absolute": float(np.abs(s).mean()),
        "observed_constant": bool(np.ptp(s) <= 1e-12), "structural_claim": "not established by observed constancy",
        "uncertainty": patient_intervals(patients, lambda rows: [(s[rows] > 0).mean(), s[rows].mean(), np.abs(s[rows]).mean()], replicates=replicates)}
    if year <= 2014:
        y, p = np.asarray(labels), np.asarray(probabilities)
        result["class_associations"] = []
        for name, mask in (("all", np.ones(len(s), dtype=bool)), ("deaths", y == 1), ("survivors", y == 0)):
            loss, correct = (p[mask] - y[mask])**2, ((p[mask] >= .5) == y[mask]).astype(float)
            pairs = [(signal, target) for signal in (s[mask], np.abs(s[mask])) for target in (loss, correct)]
            associations = [correlation_summary(signal, target) for signal, target in pairs]
            statistic = lambda rows: [np.nan if (value := correlation_summary(signal[rows], target[rows])["pearson"]) is None else value for signal, target in pairs]
            result["class_associations"].append({"class": name, "n": int(mask.sum()), "associations": associations,
                "order": "signed-loss/signed-correctness/absolute-loss/absolute-correctness",
                "uncertainty": patient_intervals(np.asarray(patients)[mask], statistic, replicates=replicates) if any(item["pearson"] is not None for item in associations) else {"status": "insufficient nonconstant support"}})
    return result

def intervention_scores(continuation, embeddings, cached_p, cached_gradients, component, patients, year, labels=None, replacement=None, *, replicates=500):
    from concept_subgroup_drift_pilot import batched_death_probabilities, batched_death_gradients
    chosen = np.array(sorted(range(len(patients)), key=lambda i: canonical_hash([str(patients[i]), CFG["seed"]]))[:256], dtype=int)
    if not len(chosen):
        return {"status": "empty subgroup"}
    h, p = np.asarray(embeddings)[chosen], np.asarray(cached_p)[chosen]
    replay = batched_death_probabilities(continuation, h, CFG["inference_batch"])
    np.testing.assert_allclose(replay, p, rtol=CFG["probability_rtol"], atol=CFG["probability_atol"])
    gradient = batched_death_gradients(continuation, h, CFG["gradient_batch"])
    np.testing.assert_allclose(gradient, np.asarray(cached_gradients)[chosen], rtol=CFG["gradient_rtol"], atol=CFG["gradient_atol"])
    variants = embedding_interventions(h, np.asarray(component)[chosen], None if replacement is None else np.asarray(replacement)[chosen])
    outputs = {}
    names = ["removed", "removal_norm_control"] + (["substituted", "substitution_norm_control"] if replacement is not None else [])
    for name in names:
        changed = batched_death_probabilities(continuation, variants[name], CFG["inference_batch"])[:, 1]
        dp = changed - p[:, 1]; dl = None if year > 2014 else (changed - np.asarray(labels)[chosen])**2 - (p[:, 1] - np.asarray(labels)[chosen])**2
        outputs[name] = {"probability_change": float(dp.mean()), "brier_change": float(dl.mean()) if dl is not None else None,
            "uncertainty": patient_intervals(np.asarray(patients)[chosen], lambda rows: [dp[rows].mean()] + ([dl[rows].mean()] if dl is not None else []), replicates=replicates),
            "classes": [{"class": outcome, "n": int((np.asarray(labels)[chosen] == outcome).sum()),
                "brier_change": float(dl[np.asarray(labels)[chosen] == outcome].mean()) if (np.asarray(labels)[chosen] == outcome).any() else None} for outcome in (0, 1)] if dl is not None else []}
    return {"status": "available", "n": len(chosen), "selection": "all or deterministic patient hash; no outcome/risk/activation rank",
            "effects": outputs, "substitution_status": "scored" if replacement is not None else "requires frozen correspondence", "interpretation": "positive removal Brier change means removal hurt predictions"}

def performance_over_time(population, temporal, cache, reference, device, *, interventions=False, replacements=None, replicates=500, threshold=None):
    from concept_subgroup_drift_pilot import load_continuation
    state, output = temporal["state"], []
    rows, predictor, roles = state["rows"], state["predictor"], state["roles"]
    continuation = load_continuation(cache, reference, device) if interventions else None
    for concept in temporal["families"]:
        family, year = concept["family"], concept["year"]
        frozen = state["references"][family]; seed, factor = map(int, frozen["medoid"])
        positions = np.flatnonzero((population["year"][rows] == year) & (roles["row_role"][rows] == "evaluation"))
        X, values = population["X"][rows[positions]], state["codes"][seed][positions, factor]
        A, R = values > frozen["cutoff"], frozen["rule"].mask(X)
        cavs = {key: np.asarray(record["cav"]) for key, record in frozen["cavs"].items() if record.get("valid") and record.get("cav") is not None}
        for idea, mask in (("idea1", A), ("idea2", R)):
            if not frozen["eligible"][idea]:
                continue
            selected = positions[mask]; ids = population["patient_id"][rows[selected]]
            p = predictor["probabilities"][selected, 1]; labels = population["y"][rows[selected]] if year <= 2014 else None
            baseline_row = next((item for item in state["performance"]["rows"] if item["family_id"] == family and item["idea"] == idea), None)
            if baseline_row and baseline_row["reference_identity"] != frozen["identity"]:
                raise ValueError("Performance baseline belongs to a changed reference")
            q = baseline_row["baseline"]["q"] if baseline_row else None
            influence = {str(key): sensitivity_summary(predictor["gradients"][selected] @ cav, ids, labels, p, year, replicates=replicates) for key, cav in cavs.items()}
            row = {"reference": reference, "family": family, "subgroup": "A" if idea == "idea1" else "R", "year": year,
                "track": "original frozen R1", "metrics": mortality_metrics(labels, p, q, year, threshold), "influence": influence,
                "activation_prevalence": concept["A_prevalence"], "activation_magnitude": concept["A_magnitude"],
                "rule_f2": concept["frozen_rule_fidelity"]["f2"] if concept["frozen_rule_fidelity"] else None, "retention": concept["retention"]}
            def performance_statistic(indices):
                loss = float(np.mean((p[indices] - labels[indices])**2))
                base = float(np.mean((q - labels[indices])**2)) if q is not None else None
                return [loss, 1 - loss / base if base is not None and base > 0 else np.nan]
            row["performance_uncertainty"] = patient_intervals(ids, performance_statistic, replicates=replicates) if year <= 2014 and len(ids) else {"status": "empty or invalid mortality year"}
            alignments = [candidate["pearson"] for dictionary in concept["yearly_representations"] for member in dictionary.get("members", []) for candidate in member["candidates"][:1]]
            row["yearly_alignment"] = float(np.mean(alignments)) if alignments else None
            for key, field in (("tcav", "tcav"), ("signed_sensitivity", "signed"), ("absolute_sensitivity", "absolute")):
                valid = [item[field] for item in influence.values() if item[field] is not None]
                row[key] = float(np.mean(valid)) if valid else None
            row["_private"] = {"patients": population["patient_id"][rows[positions]], "mask": mask, "A": A, "R": R,
                "values": values, "sensitivity": np.asarray([predictor["gradients"][positions] @ cav for cav in cavs.values()]) if cavs else None,
                "p": predictor["probabilities"][positions, 1], "y": population["y"][rows[positions]] if year <= 2014 else None}
            if interventions and len(selected):
                unit = cache.load(f"S/{seed}/{reference}")
                component = group_contribution(state["codes"][seed][selected, factor:factor + 1], unit["encoder_weight"][[factor]], unit["scaler_scale"])
                replacement = (replacements or {}).get((family, year, row["subgroup"]))
                row["intervention"] = intervention_scores(continuation, predictor["embeddings"][selected], predictor["probabilities"][selected], predictor["gradients"][selected], component, ids, year, labels, replacement, replicates=replicates)
            output.append(row)
    return output

def temporal_associations(annual, *, replicates=500):
    features = ("tcav", "signed_sensitivity", "absolute_sensitivity", "activation_prevalence", "activation_magnitude", "rule_f2", "retention", "yearly_alignment")
    keys = sorted({(r["reference"], r["family"], r["subgroup"]) for r in annual})
    output, between = [], []
    for key in keys:
        records = sorted([r for r in annual if (r["reference"], r["family"], r["subgroup"]) == key and r["year"] <= 2014], key=lambda r: r["year"])
        def series(feature, sampled=None):
            pairs, offset = [], 0
            for record in records:
                private = record["_private"]; n = len(private["patients"])
                pick = np.arange(n) if sampled is None else sampled[(sampled >= offset) & (sampled < offset + n)] - offset
                offset += n; chosen = pick[private["mask"][pick]]
                loss = float(np.mean((private["p"][chosen] - private["y"][chosen])**2)) if len(chosen) else None
                sensitivity = private["sensitivity"][:, chosen].reshape(-1) if private["sensitivity"] is not None else np.array([])
                values = {"activation_prevalence": float(private["A"][pick].mean()) if len(pick) else None,
                    "activation_magnitude": float(private["values"][pick].mean()) if len(pick) else None,
                    "rule_f2": fidelity(private["A"][pick], private["R"][pick])["f2"],
                    "retention": float(private["A"][pick][private["R"][pick]].mean()) if private["R"][pick].any() else None,
                    "tcav": float((sensitivity > 0).mean()) if len(sensitivity) else None,
                    "signed_sensitivity": float(sensitivity.mean()) if len(sensitivity) else None,
                    "absolute_sensitivity": float(np.abs(sensitivity).mean()) if len(sensitivity) else None,
                    "yearly_alignment": record["yearly_alignment"]}
                pairs.append((record["year"], loss, values[feature]))
            return pairs
        def correlations(feature, sampled=None):
            pairs = series(feature, sampled)
            levels = correlation_summary([r[1] for r in pairs], [r[2] for r in pairs])
            consecutive = [(b[1] - a[1], b[2] - a[2]) for a, b in zip(pairs, pairs[1:]) if b[0] == a[0] + 1 and None not in (a[1], b[1], a[2], b[2])]
            changes = correlation_summary([p[0] for p in consecutive], [p[1] for p in consecutive])
            return levels, changes
        patients = np.concatenate([r["_private"]["patients"] for r in records]) if records else np.array([])
        for feature in features:
            levels, changes = correlations(feature)
            statistic = lambda indices: [np.nan if result["pearson"] is None else result["pearson"] for result in correlations(feature, indices)]
            uncertainty = patient_intervals(patients, statistic, replicates=replicates) if levels["pearson"] is not None or changes["pearson"] is not None else {"status": "unavailable: insufficient nonconstant trajectory"}
            output.append({"reference": key[0], "family": key[1], "subgroup": key[2], "feature": feature, "levels": levels, "consecutive_changes": changes, "patient_uncertainty": uncertainty})
        usable = [r for r in records if r["metrics"]["brier"] is not None]
        between.append({"key": key, "mean_brier": float(np.mean([r["metrics"]["brier"] for r in usable])) if usable else None,
            **{feature: float(np.mean([r[feature] for r in usable if r[feature] is not None])) if any(r[feature] is not None for r in usable) else None for feature in features}})
    return {"within_concept": output, "between_concept_descriptive": between,
            "between_correlations_descriptive": {subgroup: {feature: correlation_summary([r["mean_brier"] for r in between if r["key"][2] == subgroup], [r[feature] for r in between if r["key"][2] == subgroup]) for feature in features} for subgroup in ("A", "R")},
            "qualification": "patient-cluster conditional uncertainty; alignment CI conditional on frozen fitting-patient matching; fitting/CAV-seed variation separate; overlapping concepts/shared patients not independent; no causal or unadjusted significance claims"}


In [ ]:
def focused_performance_checks():
    result = mortality_metrics([0, 1], [.2, .8], .5, 2014)
    assert abs(result["brier"] - .04) < 1e-12 and abs(result["skill"] - .84) < 1e-12
    assert mortality_metrics([0, 0], [.1, .2], 0., 2014)["skill"] is None
    assert mortality_metrics(None, [.1, .2], .5, 2015)["brier"] is None
    assert mortality_metrics([0, 1], [.2, .8], .5, 2014, .9)["at_frozen_threshold"]["tp"] == 0
    assert frozen_threshold([0, 1, 1, 0, 1], [.9, .9, .6, .6, .2])["threshold"] == .2
    tie = frozen_threshold([1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0], [.9, .5, .5, .5, .5, .1, .1, .1, .1, .1, .1])
    assert tie["threshold"] == .9 and tie["death_f1"] == .5 and tie["precision"] == 1.
    assert frozen_threshold([0, 0], [.1, .2])["threshold"] is None
    assert correlation_summary([1, 1, 1], [1, 2, 3])["pearson"] is None
    from concept_subgroup_drift_pilot import batched_death_probabilities, batched_death_gradients
    class LinearHead:
        def __init__(self):
            self.head = torch.nn.Linear(2, 2, bias=False); self.classes = (0, 1)
            with torch.no_grad(): self.head.weight.copy_(torch.tensor([[0., 0.], [.2, -.1]]))
        def probabilities(self, h): return torch.softmax(self.head(h), dim=1)
        def death_gradient(self, h):
            with torch.enable_grad():
                h = h.detach().clone().requires_grad_(True)
                return torch.autograd.grad(self.probabilities(h)[:, 1].sum(), h)[0].detach()
    head, h = LinearHead(), np.array([[1., 2.], [2., 1.], [0., 0.]], dtype=np.float32)
    p, gradients = batched_death_probabilities(head, h), batched_death_gradients(head, h)
    effects = intervention_scores(head, h, p, gradients, np.ones_like(h) * .1, ["a", "b", "c"], 2014, np.array([0, 1, 0]), replicates=4)
    assert set(effects["effects"]) == {"removed", "removal_norm_control"}
    from semantic_rules import RuleSet, Rule, Condition
    years = [2007, 2008, 2010, 2011, 2015]
    rule = RuleSet((Rule("r", (Condition(0, "x", ">", 0.),)),))
    frozen = {"medoid": [42, 0], "rule": rule, "cutoff": 1., "identity": "same", "eligible": {"idea1": True, "idea2": True}, "cavs": {42: {"valid": True, "cav": np.array([1., 0.])}, 43: {"valid": True, "cav": np.array([-.5, 0.])}}}
    population = {"X": np.ones((10, 1)), "year": np.repeat(years, 2), "patient_id": np.tile(["a", "b"], 5), "y": np.tile([0, 1], 5)}
    probability = np.array([.1, .8, .2, .7, .3, .6, .4, .5, .25, .75])
    state = {"rows": np.arange(10), "roles": {"row_role": np.array(["evaluation"] * 10)}, "references": {"f": frozen}, "codes": {42: np.tile([[0.], [3.]], (5, 1))},
        "predictor": {"probabilities": np.column_stack([1 - probability, probability]), "gradients": np.tile([[1., 0.], [-1., 0.]], (5, 1))},
        "performance": {"rows": [{"family_id": "f", "idea": idea, "reference_identity": "same", "baseline": {"q": .5}} for idea in ("idea1", "idea2")]}}
    concepts = [{"family": "f", "year": year, "A_prevalence": .5, "A_magnitude": 1.5, "frozen_rule_fidelity": {"f2": 5 / 6}, "retention": .5, "yearly_representations": []} for year in years]
    annual = performance_over_time(population, {"state": state, "families": concepts}, None, 2007, "cpu", replicates=4)
    associations = temporal_associations(annual, replicates=4)
    assert len(annual) == 10 and all(item["consecutive_changes"]["n"] == 2 for item in associations["within_concept"] if item["feature"] == "activation_prevalence")
    assert all(row["tcav"] == .5 for row in annual) and annual[-1]["metrics"]["brier"] is None
    assert (annual[0]["_private"]["sensitivity"][:, annual[0]["_private"]["mask"]] > 0).mean() == .5
    return {"status": "passed", "scope": "Brier/skill/2015 exclusion/frozen threshold tie-breaks/constant association/linear-head exact replay/intervention"}


In [ ]:
# Report: frozen predictor and A/R subgroup performance, influence and associations over time. Outcome claims use years ≤ 2014.
def predictor_frame():
    rows, bins = [], []
    for variant, run in RUNS.items():
        for year, entries in run["predictor"].items():
            for entry in entries:
                metrics = entry.get("metrics") or {}
                frozen, oracle = metrics.get("at_frozen_threshold") or {}, entry.get("oracle_diagnostic") or {}
                rows.append({"variant": variant, "reference": entry["reference"], "year": entry["year"], "lag": entry["year"] - entry["reference"],
                             "auroc": entry.get("auroc"), "average_precision": entry.get("average_precision"), "brier": metrics.get("brier"),
                             "calibration_gap": metrics.get("calibration_gap"), "mortality": metrics.get("mortality"), "mean_risk": metrics.get("mean_risk"),
                             "frozen_f1": frozen.get("f1"), "frozen_sensitivity": frozen.get("sensitivity"), "frozen_specificity": frozen.get("specificity"),
                             "oracle_f1": oracle.get("death_f1"), "rows": metrics.get("rows")})
                bins += [{"variant": variant, "reference": entry["reference"], "lag": entry["year"] - entry["reference"], **item} for item in metrics.get("calibration_bins", [])]
    return pd.DataFrame(rows), pd.DataFrame(bins)

def annual_frame():
    rows, associations = [], []
    for variant, run in RUNS.items():
        for reference, export in run["evaluate"].items():
            for row in export["annual"]:
                metrics, effects = row.get("metrics") or {}, ((row.get("intervention") or {}).get("effects") or {})
                removed, control = effects.get("removed") or {}, effects.get("removal_norm_control") or {}
                intervals = (removed.get("uncertainty") or {}).get("intervals") or []
                interval = intervals[1] if len(intervals) > 1 else None  # order: probability change, Brier change
                rows.append({"variant": variant, "reference": reference, "family": row["family"], "subgroup": row["subgroup"], "year": row["year"],
                             "lag": row["year"] - reference, "brier": metrics.get("brier"), "skill": metrics.get("skill"), "rows": metrics.get("rows"),
                             "tcav": row.get("tcav"), "signed_sensitivity": row.get("signed_sensitivity"), "absolute_sensitivity": row.get("absolute_sensitivity"),
                             "activation_prevalence": row.get("activation_prevalence"), "removal_brier_change": removed.get("brier_change"),
                             "control_brier_change": control.get("brier_change"),
                             "removal_ci_excludes_zero": bool(interval and (interval[0] > 0 or interval[1] < 0)) if interval else None})
            for subgroup, correlations in export["associations"]["between_correlations_descriptive"].items():
                associations += [{"variant": variant, "reference": reference, "subgroup": subgroup, "feature": feature, "pearson": value.get("pearson"), "n": value.get("n")}
                                 for feature, value in correlations.items()]
    return pd.DataFrame(rows), pd.DataFrame(associations)

def show_performance():
    predictor, bins = predictor_frame()
    outcome = predictor[predictor.year <= LAST_OUTCOME]
    for key, cmap in (("auroc", "viridis"), ("brier", "magma_r")):
        fig, axes = plt.subplots(1, len(ORDER), figsize=(6.4 * len(ORDER), 4.6), squeeze=False)
        for ax, variant in zip(axes[0], ORDER):
            heat = outcome[outcome.variant == variant].pivot_table(index="reference", columns="year", values=key)
            heatmap(heat, ax, cmap=cmap, vmin=outcome[key].min(), vmax=outcome[key].max(), cbar=False)
            ax.set_title(f"{variant}: frozen predictor {key} (years ≤ {LAST_OUTCOME})")
        save(fig, f"predictor_{key}")
    if "baseline" in RUNS and len(ORDER) > 1:
        base = outcome[outcome.variant == "baseline"].set_index(["reference", "year"])
        fig, axes = plt.subplots(1, len(ORDER) - 1, figsize=(6.4 * (len(ORDER) - 1), 4.6), squeeze=False)
        for ax, variant in zip(axes[0], [variant for variant in ORDER if variant != "baseline"]):
            delta = (outcome[outcome.variant == variant].set_index(["reference", "year"]).auroc - base.auroc).reset_index()
            heatmap(delta.pivot_table(index="reference", columns="year", values="auroc"), ax, fmt="+.2f", cmap="vlag", center=0, cbar=False)
            ax.set_title(f"AUROC: {variant} − baseline")
        save(fig, "predictor_auroc_delta")
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.4))
    sns.lineplot(outcome, x="lag", y="auroc", hue="variant", palette=PALETTE, hue_order=ORDER, estimator="median", errorbar=("pi", 50), marker="o", ax=axes[0])
    axes[0].set_title("AUROC by years after reference (median, IQR)")
    sns.lineplot(outcome, x="lag", y="calibration_gap", hue="variant", palette=PALETTE, hue_order=ORDER, estimator="median", marker="o", ax=axes[1], legend=False)
    axes[1].set_title("Calibration gap by years after reference")
    f1 = outcome[outcome.lag == 0].melt(id_vars=["variant", "reference"], value_vars=["frozen_f1", "oracle_f1"], var_name="threshold", value_name="death_f1")
    sns.barplot(f1, x="variant", y="death_f1", hue="threshold", order=ORDER, ax=axes[2])
    axes[2].set_title("Death-F1, year = reference:\nfrozen vs oracle threshold")
    calibration = bins[(bins.lag == 0) & (bins.n > 0)].dropna(subset=["risk", "mortality"])
    pooled = calibration.assign(weighted_risk=lambda d: d.risk * d.n, weighted_mortality=lambda d: d.mortality * d.n).groupby(["variant", "bin"])[
        ["weighted_risk", "weighted_mortality", "n"]].sum()
    pooled = pooled.assign(risk=pooled.weighted_risk / pooled.n, mortality=pooled.weighted_mortality / pooled.n).reset_index()
    sns.lineplot(pooled, x="risk", y="mortality", hue="variant", palette=PALETTE, hue_order=ORDER, marker="o", ax=axes[3], legend=False)
    axes[3].plot([0, .6], [0, .6], color="grey", ls="--", lw=1)
    axes[3].set_title("Reliability, year = reference (pooled bins)")
    save(fig, "predictor_summary")
    show(outcome.groupby(["variant", "lag"])[["auroc", "average_precision", "brier", "frozen_f1", "oracle_f1"]].median().unstack("variant").round(3), "predictor_by_lag")
    annual, associations = annual_frame()
    outcome = annual[annual.year <= LAST_OUTCOME]
    grid = sns.catplot(outcome, x="lag", y="skill", hue="variant", col="subgroup", kind="box", palette=PALETTE, hue_order=ORDER, height=3.8, aspect=1.6, fliersize=1.5)
    for ax in grid.axes.flat:
        ax.axhline(0, color="grey", lw=1)
        ax.set_ylim(-1.5, 1)
    grid.figure.suptitle("Subgroup Brier skill vs frozen reference baseline (A = factor cohort, R = rule cohort)", y=1.04)
    save(grid.figure, "subgroup_skill")
    influence = annual.melt(id_vars=["variant", "subgroup", "lag"], value_vars=["tcav", "absolute_sensitivity", "removal_brier_change"], var_name="measure")
    grid = sns.relplot(influence, x="lag", y="value", hue="variant", style="subgroup", col="measure", kind="line", palette=PALETTE, hue_order=ORDER,
                       estimator="median", errorbar=None, marker="o", height=3.6, aspect=1.25, facet_kws={"sharey": False})
    grid.figure.suptitle("Influence through time: TCAV, |sensitivity| and Brier change on removal (median)", y=1.04)
    save(grid.figure, "influence_by_lag")
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    sns.scatterplot(annual, x="control_brier_change", y="removal_brier_change", hue="variant", style="subgroup", palette=PALETTE, hue_order=ORDER, s=14, alpha=.5, ax=axes[0])
    low = np.nanmin(annual[["control_brier_change", "removal_brier_change"]].values)
    high = np.nanmax(annual[["control_brier_change", "removal_brier_change"]].values)
    axes[0].plot([low, high], [low, high], color="grey", ls="--", lw=1)
    axes[0].set_title("Concept removal vs norm-matched control (Brier change; > 0 = removal hurts)")
    share = annual.groupby(["variant", "subgroup"]).removal_ci_excludes_zero.mean().reset_index()
    sns.barplot(share, x="variant", y="removal_ci_excludes_zero", hue="subgroup", order=ORDER, ax=axes[1])
    axes[1].set(title="Share of concept-years whose removal Brier-change CI excludes 0", ylim=(0, 1))
    save(fig, "influence_removal")
    heat = associations.groupby(["variant", "subgroup", "feature"]).pearson.median().unstack("feature").reindex(ORDER, level="variant")
    fig, ax = plt.subplots(figsize=(12, 0.5 * len(heat) + 1.5))
    heatmap(heat, ax, size=9, cmap="vlag", center=0, vmin=-1, vmax=1)
    ax.set_title("Between-concept association with subgroup Brier (median Pearson over references; descriptive)")
    save(fig, "associations")

if MODE == "report":
    show_performance()

## Secondary prospective question — disabled by default

After the two main ideas and concurrent findings are complete, first report
feasible consecutive-year transitions through 2014. Freeze one prespecified
concept signal and the earliest calendar boundary with sufficient past support
and full design rank before prediction scores are computed. Compare no-change,
performance history and history plus that signal for next-year Brier change.
Scalers fit on past training transitions only. Activation requires the completed
main report, matching design freeze and a declared budget. Unavailable support
is an acceptable finding; this remains an exploratory R1 continuation.


In [ ]:
def secondary_prospective(annual, signal, *, enabled=False, freeze=None, main_results_complete=False, budget_seconds=None):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    allowed = {"tcav", "signed_sensitivity", "absolute_sensitivity", "activation_prevalence", "activation_magnitude", "rule_f2", "retention", "yearly_alignment"}
    if signal not in allowed:
        raise ValueError("One prespecified concept signal required; no composite or outcome-driven selection")
    groups = {}
    for row in annual:
        if row["year"] <= 2014:
            group = groups.setdefault((row["reference"], row["family"], row["subgroup"]), {})
            if row["year"] in group:
                raise ValueError("Duplicate annual concept/subgroup/year; score variants separately")
            group[row["year"]] = row
    designs, audit = {}, []
    for key, years in groups.items():
        transitions = [(year, years[year]["metrics"]["brier"], years[year].get(signal), years[year + 1]["metrics"]["brier"])
            for year in sorted(years) if year + 1 in years and year + 1 <= 2014]
        usable = [row for row in transitions if all(value is not None and np.isfinite(value) for value in row[1:])]
        designs[key] = usable
        audit.append({"key": key, "consecutive_transitions": len(transitions), "usable": len(usable),
            "missing_or_invalid": len(transitions) - len(usable), "years": [row[0] for row in usable]})
    def eligible(rows, boundary):
        train = [row for row in rows if row[0] < boundary]; test = [row for row in rows if row[0] >= boundary]
        rank = int(np.linalg.matrix_rank(np.array([[1., row[1], row[2]] for row in train]))) if train else 0
        return len(train) >= 5 and len(test) >= 2 and rank == 3, {"n_train": len(train), "n_test": len(test), "design_rank": rank}
    candidates = sorted({row[0] for rows in designs.values() for row in rows})
    boundary = next((year for year in candidates if any(eligible(rows, year)[0] for rows in designs.values())), None)
    design_hash = canonical_hash({str(key): [[row[0], row[1], row[2]] for row in rows] for key, rows in designs.items()})
    proposed = {"boundary": boundary, "signal": signal, "design_hash": design_hash}
    output = {"status": "disabled" if not enabled else "unavailable", "feasibility": audit, "proposed_freeze": proposed,
        "support_policy": "at least 5 past training transitions, 2 held-out transitions, intercept/history/signal rank3",
        "label_timing": "performance/concepts available through end-year t; forecast t+1; latest training target year <= forecast origin year",
        "qualification": "secondary exploratory calendar holdout; examined R1 2014/2015 are not clean new confirmation; concepts share patients", "results": []}
    if boundary is None:
        output["reason"] = "insufficient chronological support or constant/rank-deficient design"
        return output
    output["boundary_support"] = [{"key": key, **eligible(rows, boundary)[1]} for key, rows in designs.items()]
    if not enabled:
        return output
    if not main_results_complete or freeze != proposed or budget_seconds is None or budget_seconds <= 0:
        raise ValueError("Require completed main findings, exact frozen boundary/signal/design and declared positive budget")
    started = time.monotonic()
    for key, rows in designs.items():
        if not eligible(rows, boundary)[0]:
            output["results"].append({"key": key, "status": "insufficient support at frozen calendar boundary"}); continue
        if time.monotonic() - started >= budget_seconds:
            raise TimeoutError("Secondary prospective budget exhausted")
        train, test = [row for row in rows if row[0] < boundary], [row for row in rows if row[0] >= boundary]
        X, y = np.array([[row[1], row[2]] for row in train]), np.array([row[3] - row[1] for row in train])
        future, target = np.array([[row[1], row[2]] for row in test]), np.array([row[3] - row[1] for row in test])
        predictions = {"no_change": np.zeros(len(test))}
        for name, columns in (("performance_history", [0]), ("history_plus_signal", [0, 1])):
            model = make_pipeline(StandardScaler(), Ridge(alpha=1.)).fit(X[:, columns], y)
            predictions[name] = model.predict(future[:, columns])
        output["results"].append({"key": key, "status": "exploratory", "test_years": [row[0] + 1 for row in test],
            "fit_fingerprint": canonical_hash({"X": X.tolist(), "y": y.tolist()}),
            "predicted_brier_change": {name: value.tolist() for name, value in predictions.items()},
            "scores": {name: {"mae": float(np.abs(value - target).mean()), "rmse": float(np.sqrt(np.mean((value - target)**2)))} for name, value in predictions.items()}})
    output["status"] = "completed exploratory secondary analysis"
    return output


In [ ]:
def focused_prospective_checks():
    import copy
    losses = [.1, .14, .11, .18, .13, .17, .12, .16, None]
    signals = [.2, .1, .4, .2, .6, .3, .8, .5, 999.]
    annual = [{"reference": 2007, "family": "synthetic", "subgroup": "A", "year": 2007 + i,
        "metrics": {"brier": loss}, "tcav": signals[i]} for i, loss in enumerate(losses)]
    feasibility = secondary_prospective(annual, "tcav")
    assert feasibility["status"] == "disabled" and feasibility["results"] == []
    assert feasibility["proposed_freeze"]["boundary"] == 2012 and feasibility["feasibility"][0]["usable"] == 7
    options = {"enabled": True, "freeze": feasibility["proposed_freeze"], "main_results_complete": True, "budget_seconds": 30}
    before = secondary_prospective(annual, "tcav", **options)
    poisoned = copy.deepcopy(annual); poisoned[7]["metrics"]["brier"] = .99; poisoned[8]["metrics"]["brier"] = .99
    after = secondary_prospective(poisoned, "tcav", **options)
    assert before["results"][0]["fit_fingerprint"] == after["results"][0]["fit_fingerprint"]
    assert before["results"][0]["predicted_brier_change"] == after["results"][0]["predicted_brier_change"]
    assert max(before["results"][0]["test_years"]) == 2014
    try:
        secondary_prospective(annual + [annual[0]], "tcav")
    except ValueError:
        pass
    else:
        raise AssertionError("Duplicate annual rows must fail")
    return {"status": "passed", "scope": "disabled default/chronological rank-support freeze/future poisoning/2015 exclusion"}


In [ ]:
# Report: secondary prospective question stays disabled; show only the design support it would have.
def show_prospective():
    rows = [{"variant": variant, "reference": item["key"][0], "subgroup": item["key"][2], "design_rank": item["design_rank"],
             "n_train": item["n_train"], "n_test": item["n_test"]}
            for variant, run in RUNS.items() for item in (run["prospective"] or {}).get("boundary_support", [])]
    print({variant: run["evaluate_metrics"]["prospective_status"] for variant, run in RUNS.items()})
    if rows:
        frame = pd.DataFrame(rows)
        show(frame.groupby(["variant", "subgroup"])[["design_rank", "n_train", "n_test"]].describe().loc[:, (slice(None), ["min", "50%", "max"])].round(1),
             "prospective_support")

if MODE == "report":
    show_prospective()

In [ ]:
def focused_setup_checks():
    assert resolve_device("cpu", cuda_available=False) == "cpu"
    assert resolve_device("auto", cuda_available=True) == "cuda"
    for requested, available, expected in [("tpu", False, ValueError),
                                           ("cuda", False, RuntimeError)]:
        try:
            resolve_device(requested, cuda_available=available)
        except expected:
            pass
        else:
            raise AssertionError(f"Device {requested!r} must fail explicitly")
    capture = io.StringIO()
    handler = logging.StreamHandler(capture)
    log.addHandler(handler)
    try:
        assert resolve_device("auto", cuda_available=False) == "cpu"
    finally:
        log.removeHandler(handler)
    assert "CUDA unavailable" in capture.getvalue()
    assert ordered_rows_hash(["p1:2007", "p2:2007"]) != ordered_rows_hash(["p2:2007", "p1:2007"])
    try:
        ordered_rows_hash(["same", "same"])
    except ValueError:
        pass
    else:
        raise AssertionError("Duplicate rows must fail")
    parts = dict.fromkeys(IDENTITY_FIELDS, "synthetic-fixture")
    base = object_identity(parts)
    assert object_identity(dict(reversed(list(parts.items())))) == base
    for key in IDENTITY_FIELDS:
        changed = dict(parts, **{key: "changed"})
        assert object_identity(changed) != base, f"Identity ignores {key}"
        missing = dict(parts)
        del missing[key]
        try:
            object_identity(missing)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Missing identity field accepted: {key}")
    assert CFG["primary_outcome_last_year"] == 2014
    assert CFG["prospective_enabled"] is False
    assert CFG["amp"] == "off" and CFG["tf32"] is False
    return {"status": "passed", "scope": "setup/device/ordered-row/object identity"}


## Run modes and compact exports

`v1` checks real encoding, replay/gradients, removal, identity, domain inference and
one 5-epoch fit on at most 128 rows. `develop` reads reference fitting/tuning patients
for the redesigned rules/models (baseline plus three one-setting policies, counts versus
presence), runs the retrospective six-check diagnosis, and freezes one recipe by a
prespecified rule. Develop also compares the original rule-discovery allocation with
one larger candidate on common held-out tune patients and freezes one Death-F1
threshold per reference from year-R non-context, non-evaluation rows (option C).
`evaluate` verifies that recipe hash and variant, then runs the final suite once.
Exports are aggregate JSON: private keys, identifiers, fitted objects and long
per-patient arrays are summarized or omitted and stay on Fiji.


In [ ]:
REFERENCES, PRIMARY = tuple(range(2007, 2016)), tuple(range(2007, 2014))
THRESHOLD_ROLES = ("sae_discovery", "rule_discovery", "cav_fit", "concept_selection")
POLICY_CHANGES = ({"cap": 40}, {"clustering": .5}, {"recurrence": .3})
TRANSFORMS = ({"kind": "counts"}, {"kind": "presence"})  # aggregate: no recorded clinical definition/source
DENY = {"models", "frozen", "rule", "state", "codes", "fit", "tune", "checkpoints", "reconstructed"}

def compact(value):
    # Aggregate export only: private keys, fitted objects and long per-patient arrays stay on Fiji.
    if isinstance(value, dict):
        return {str(k): compact(v) for k, v in value.items() if not str(k).startswith("_") and k not in DENY}
    if isinstance(value, (list, tuple)):
        if len(value) > 32 and all(isinstance(v, str) for v in value):
            return {"omitted": "identifier list", "n": len(value)}
        return [compact(v) for v in value]
    if isinstance(value, np.ndarray):
        if value.dtype.kind in "OUS":
            return {"omitted": "identifier array", "n": int(value.size)}
        if value.size <= 32:
            return compact(value.tolist())
        finite = value[np.isfinite(value)].astype(float) if value.dtype.kind in "fc" else value.ravel().astype(float)
        summary = dict(zip(("q10", "median", "q90"), np.quantile(finite, [.1, .5, .9]).tolist())) if finite.size else {}
        return {"shape": list(value.shape), "nonfinite": int(value.size - finite.size), "mean": float(finite.mean()) if finite.size else None, **summary}
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    return value if value is None or isinstance(value, str) else {"omitted": type(value).__name__}

STAGES = {}
def stage(name, function, *args, **kwargs):
    started = time.monotonic()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        result = function(*args, **kwargs)
        STAGES[name] = {"status": result.get("status", "ok") if isinstance(result, dict) else "ok"}
        return result
    except (ValueError, RuntimeError, KeyError, IndexError) as error:
        if "identity" in str(error) or "replay" in str(error).lower() or "Nonfinite" in str(error):
            raise  # Stop conditions: never recorded as an ordinary failed unit.
        STAGES[name] = {"status": "failed", "reason": f"{type(error).__name__}: {error}"}
        return {"status": "failed", "reason": STAGES[name]["reason"]}
    finally:
        STAGES[name].update(seconds=time.monotonic() - started, **({"peak_allocated": torch.cuda.max_memory_allocated(),
            "peak_reserved": torch.cuda.max_memory_reserved()} if torch.cuda.is_available() else {}))
        atomic_json(OUT / "stages.json", STAGES)

def export(name, value):
    path = OUT / f"{name}.json"
    atomic_json(path, compact(value))
    return path.name

def member_rows(population, state, reference):
    import concept_subgroup_drift_pilot as pilot
    rules_unit = state["rules_unit"]
    rows, role = state["rows"], state["roles"]["row_role"][state["rows"]]
    rule = np.flatnonzero((population["year"][rows] == reference) & (role == "rule_discovery"))
    if not np.array_equal(rows[rule], rules_unit["rule_row_index"]):
        raise ValueError("Rule-row identity differs from frozen R1 rules unit")
    evaluation = np.flatnonzero((population["year"][rows] == reference) & (role == "evaluation"))
    return rule, evaluation, pilot

def inner_split(patients, rule, state, config, reference, family_id, allocation="original"):
    import concept_subgroup_drift_pilot as pilot
    seed, factor = map(int, state["references"][family_id]["medoid"])
    split_seed = pilot._derived_seed("rule_inner_split", config["split_seed"], reference, family_id)
    fit_ids, tune_ids = pilot.rule_inner_split(patients[rule], state["codes"][seed][rule, factor], split_seed)
    fit, tune = rule[np.isin(patients[rule], fit_ids)], rule[np.isin(patients[rule], tune_ids)]
    if allocation == "larger":
        larger, common, _ = allocation_rows(patients, rule, fit, tune)
        fit, tune = (larger, common) if larger is not None else (fit, tune)
    return fit, tune, split_seed

def allocation_comparison(X, names, activations, cutoff, patients, rule, fit, tune, backend, selection, policy):
    """Original versus larger rule-discovery fit, same frozen target and policy, scored on common held-out tune patients."""
    larger, common, audit = allocation_rows(patients, rule, fit, tune)
    if larger is None:
        return audit
    target = activations > cutoff
    support = {"larger_fit": [int(target[larger].sum()), int((~target[larger]).sum())],
               "common_tune": [int(target[common].sum()), int((~target[common]).sum())]}
    if min(min(pair) for pair in support.values()) < CFG["allocation"]["min_target_support"]:
        return {**audit, "status": "insufficient frozen-target support", "support": support}
    scored = {name: fit_member_explanation(X, names, activations, cutoff, patients, rows, common, patients[rule], backend, selection, [policy])[0]
              for name, rows in (("original", fit), ("larger", larger))}
    return {**audit, "status": "compared", "support": support, "scored_on": "common held-out tune patients", **scored}

def reference_threshold(population, state, reference):
    """Option C: Death-F1 threshold on year-R non-context, non-evaluation rows, frozen before later years are scored."""
    if reference > CFG["primary_outcome_last_year"]:
        return {"status": "unavailable: invalid mortality recording", "threshold": None}
    rows, role = state["rows"], state["roles"]["row_role"][state["rows"]]
    at = np.flatnonzero((population["year"][rows] == reference) & np.isin(role, THRESHOLD_ROLES))
    return {**frozen_threshold(population["y"][rows[at]], state["predictor"]["probabilities"][at, 1]), "roles": list(THRESHOLD_ROLES)}

def window_probe(population, config, device, reference=2008, rows=128):
    """v3 window context on the CPU predictor route builds use (all_history at 2008) through the legacy predictor path: one TabPFN fit, one capture of <=128 evaluation rows, no outcomes read."""
    import concept_subgroup_drift_pilot as pilot
    torch.cuda.empty_cache() if device == "cuda" else None  # TabPFN capacity counts memory this process still reserves.
    seen = {}
    class Probed(Exception):
        pass
    roles = pilot.reference_year_roles(population, reference, config["split_seed"])
    def fit(frame, names, device, seed=42, **options):
        seen["context_rows"], seen["context_years"] = len(frame), sorted(int(y) for y in frame["year"].unique())
        seen.update(options)
        return pilot.fit_frozen_predictor(frame, names, device, seed=seed, **options)
    def capture(system, frame, device):
        query = np.flatnonzero((population["year"] == reference) & (roles["row_role"] == "evaluation"))[:rows]
        columns = pilot._feature_columns(population["X"].shape[1])
        out = pilot.capture_frozen_predictions(system, pilot._prediction_frame(population, query, columns), device)["probabilities"]
        out = out.detach().cpu().numpy() if hasattr(out, "detach") else np.asarray(out)
        seen.update(query_rows=len(query), finite=bool(np.isfinite(out).all()))
        raise Probed
    try:
        pilot.predictor_unit(population, reference, roles, dict(config, context_window="all_history", predictor_cpu_from=reference), device, fit=fit, capture=capture)
    except Probed:
        pass
    return {"reference": reference, "window": "all_history", **seen}


def reference_development(population, state, config, reference, recipe=None):
    """Reference-only: rules fit/tuned on the R1 inner split of rule_discovery patients; evaluation rows never touched."""
    rule, _, pilot = member_rows(population, state, reference)
    X, names, patients = population["X"][state["rows"]], list(population["feature_names"]), population["patient_id"][state["rows"]].astype(str)
    backend = pilot._backend_settings()
    selection = {key: value for key, value in pilot.rule_selection_config(config["rule"]).__dict__.items()}
    recurrence = config["rule"]["recurrence"]
    baseline = {"cap": recurrence["max_candidates_per_bootstrap"], "clustering": recurrence["family_similarity_threshold"], "recurrence": recurrence["min_family_recurrence"]}
    policies = candidate_policies(baseline, POLICY_CHANGES) if recipe is None else candidate_policies(recipe["policy"])
    transforms = TRANSFORMS if recipe is None else (recipe["transform"],)
    output = []
    for family_id, frozen in state["references"].items():
        seed, factor = map(int, frozen["medoid"])
        fit, tune, split_seed = inner_split(patients, rule, state, config, reference, family_id, recipe["allocation"] if recipe else "original")
        members = [tuple(map(int, member)) for member in state["families"][family_id]["members"]]
        values = np.column_stack([state["codes"][s][:, k] for s, k in members])
        cutoffs = member_cutoffs(values, rule, frozen["activation_target"])
        for index, (s, k) in enumerate(members):
            cutoff = frozen["cutoff"] if (s, k) == (seed, factor) else cutoffs[index]["cutoff"]
            record = {"family": family_id, "seed": s, "factor": k, "medoid": (s, k) == (seed, factor), "cutoff": cutoff, "cutoff_audit": cutoffs[index], "results": []}
            for transform in transforms if cutoff is not None else ():
                features, feature_names = explanation_features(X, names, **transform)
                fitted = stage(f"rules/{reference}/{family_id}/{s}:{k}/{transform['kind']}", fit_member_explanation, features, feature_names, values[:, index], cutoff,
                    patients, fit, tune, patients[rule], dict(backend, random_state=split_seed), selection, policies)
                record["results"].append({"transform": transform, "policies": fitted if isinstance(fitted, list) else [fitted]})
            if record["medoid"]:
                features, _ = explanation_features(X, names, **(TRANSFORMS[0] if recipe is None else recipe["transform"]))
                record["continuous"] = stage(f"continuous/{reference}/{family_id}/{s}:{k}", fit_continuous_activation, features, values[:, index], patients, fit, tune, patients[rule])
                if recipe is None and cutoff is not None:
                    counts, count_names = explanation_features(X, names, **TRANSFORMS[0])
                    record["allocation"] = stage(f"allocation/{reference}/{family_id}", allocation_comparison, counts, count_names, values[:, index], cutoff,
                        patients, rule, fit, tune, dict(backend, random_state=split_seed), selection, policies[0])
            output.append(record)
    return output

def freeze_recipe(development, thresholds):
    """Prespecified rules: highest median tune F2 over every (family, member), ties keep the R1 baseline; larger allocation
    only with positive median paired F2 gain on common tune patients; per-reference option-C thresholds."""
    scores = {}
    for record in development:
        for result in record["results"]:
            for item in result["policies"]:
                if isinstance(item, dict) and "tune" in item and item["tune"]["f2"] is not None:
                    key = json.dumps({"transform": result["transform"], "policy": item["policy"]}, sort_keys=True)
                    scores.setdefault(key, []).append(item["tune"]["f2"])
    if not scores:
        raise RuntimeError("No feasible reference development description; nothing to freeze")
    table = [{"setting": json.loads(key), "n": len(values), "median_tune_f2": float(np.median(values))} for key, values in scores.items()]
    best = max(table, key=lambda row: (round(row["median_tune_f2"], 6), row["setting"]["transform"]["kind"] == "counts" and row["setting"]["policy"] == table[0]["setting"]["policy"]))
    pairs = [(record["allocation"]["original"]["tune"]["f2"], record["allocation"]["larger"]["tune"]["f2"]) for record in development
             if isinstance(record.get("allocation"), dict) and record["allocation"].get("status") == "compared"]
    gains = [larger - original for original, larger in pairs if None not in (original, larger)]
    recipe = {"variant": CFG["variant"], "policy": best["setting"]["policy"], "transform": best["setting"]["transform"],
        "allocation": "larger" if gains and float(np.median(gains)) > 0 else "original",
        "allocation_table": {"compared": len(pairs), "scored": len(gains), "median_f2_gain": float(np.median(gains)) if gains else None, **CFG["allocation"]},
        "thresholds": {str(r): {key: value for key, value in item.items() if key in {"status", "threshold", "death_f1", "precision", "recall", "rows", "deaths", "roles", "reason"}}
                       for r, item in thresholds.items()},
        "continuous": "constant/ridge/boosting refit on reference fit rows",
        "mapping": {"ranks": [1, 2, 3], "alphas": [.1, 1., 10.], "controls": "constant/shuffled/prevalence-matched nonfamily"},
        "intervention_sample": "all or first 256 by patient hash", "bootstrap_replicates": 500, "cav_seeds": [42, 43, 44], "sae_seeds": [42, 135, 215],
        "mechanism_references": "2011/2012 post hoc plus reference-support comparator", "development_table": table}
    return {**recipe, "recipe_hash": canonical_hash(recipe)}

def disappearance_reference(population, state, cache, reference, device, *, mechanism=False):
    """Retrospective legacy diagnostic: activations/probabilities only; no outcomes are read."""
    import concept_subgroup_drift_pilot as pilot
    rows, roles, predictor = state["rows"], state["roles"], state["predictor"]
    years, role = population["year"][rows], roles["row_role"][rows]
    evaluation = role == "evaluation"
    continuation = pilot.load_continuation(cache, reference, device)
    output = {"reference": reference, "identity": [], "composition": [], "scaling": [], "reconstruction": [], "label": "retrospective legacy diagnosis"}
    for family_id, frozen in state["references"].items():
        seed, factor = map(int, frozen["medoid"])
        unit = cache.load(f"S/{seed}/{reference}")
        W, b, mu, sigma = unit["encoder_weight"], unit["encoder_bias"], unit["scaler_mean"], unit["scaler_scale"]
        h = predictor["embeddings"]
        cached = state["codes"][seed][:, factor] > frozen["cutoff"]
        parts = {"source": state["cache"].source_hash, "data": population["fingerprint"]["population_identity"],
            "ordered_rows": predictor["output_row_identity"], "roles": roles["identity"], "predictor_context": predictor["context_identity"],
            "domain": "calendar year minus reference", "scaler": unit["scaler_identity"], "dictionary": unit["sae_identity"],
            "cutoff": frozen["cutoff"], "rule": frozen["identity"], "transform": "counts", "config": state["cache"].config_hash}
        expected = dict(parts, scaler=state["dictionary_ids"][seed]["scaler_identity"], dictionary=state["dictionary_ids"][seed]["sae_identity"])
        for year in range(reference, 2016):
            at = np.flatnonzero((years == year) & evaluation)
            check = disappearance_identity(h[at], W, b, mu, sigma, factor, frozen["cutoff"], cached[at], parts, expected)
            output["identity"].append({"family": family_id, "year": year, **check})
        values = state["codes"][seed][:, factor]
        for year in range(reference + 1, 2016):
            keep = evaluation & np.isin(years, [reference, year])
            table, before, after = composition_diagnostic(population["patient_id"][rows][keep], years[keep], values[keep], reference, year)
            output["composition"].append({"family": family_id, "year": year, "tables": table})
            if len(before):
                hk = h[keep]
                scaling = scaling_diagnostic(hk[before], hk[after], W, b, mu, sigma, factor, frozen["cutoff"])
                output["scaling"].append({"family": family_id, "year": year, "paired": len(before), **{key: value for key, value in scaling.items()
                    if key in {"delta_u", "dominant_coordinates", "zero_margin_before", "zero_margin_after", "cutoff_margin_before", "cutoff_margin_after"}},
                    "delta_u_selected": scaling["delta_u"][:, factor]})
    for seed in sorted({int(item["medoid"][0]) for item in state["references"].values()}):
        unit = cache.load(f"S/{seed}/{reference}")
        predict = lambda value: pilot.batched_death_probabilities(continuation, np.asarray(value, dtype=np.float32), CFG["inference_batch"] * 64)[:, 1]
        for year in range(reference, 2016):
            at = np.flatnonzero((years == year) & evaluation)
            if len(at):
                result = reconstruction_diagnostic(predictor["embeddings"][at], state["codes"][seed][at], unit["encoder_weight"], unit["decoder_bias"], unit["scaler_mean"], unit["scaler_scale"], predict)
                output["reconstruction"].append({"seed": seed, "year": year, **result})
    if mechanism:
        output["mechanism"] = mechanism_study(population, state, reference, device)
    return output

def mechanism_study(population, state, reference, device, *, rows_per_period=256, fits=None, epochs=None, max_fit_rows=None):
    """Check 2 (2x2 frozen-context domain grid) and the bounded nine-fit dictionary study for one reference."""
    import concept_subgroup_drift_pilot as pilot
    rows, roles = state["rows"], state["roles"]
    years, role = population["year"][rows], roles["row_role"][rows]
    context_device = "cpu" if state["predictor"].get("device") == "cpu" else device  # all_history units beyond GPU capacity
    system = restore_frozen_context(population, roles, state["predictor"], context_device, int(state["config"]["split_seed"]))
    later = next((y for y in range(reference + 1, 2015) if ((years == y) & (role == "evaluation")).any()), None)
    pick = lambda year: sorted(np.flatnonzero((years == year) & (role == "evaluation")), key=lambda i: canonical_hash([str(population["patient_id"][rows][i]), CFG["seed"]]))[:rows_per_period]
    columns = pilot._feature_columns(population["X"].shape[1])
    seeds = sorted({int(item["medoid"][0]) for item in state["references"].values()})
    dictionaries = {s: pilot.load_reference_sae(state["cache"], reference, s, device) for s in seeds}
    encode = lambda h: {s: pilot.encode_with_reference_sae(d, h, device, CFG["encoding_batch"]) for s, d in dictionaries.items()}
    grid = domain_grid(system, pilot._prediction_frame(population, rows[pick(reference)], columns), pilot._prediction_frame(population, rows[pick(later)], columns), later, context_device, encode) if later else []
    medoids = {family: tuple(map(int, item["medoid"])) for family, item in state["references"].items()}
    summary = [{"features": cell["features"], "domain_year": cell["domain_year"], "rows": len(cell["embeddings"]), "identity": cell["identity"],
        "death_probability": cell["probabilities"][:, 1], "medoid_above_cutoff": {family: float((cell["activations"][s][:, k] > state["references"][family]["cutoff"]).mean()) if len(cell["embeddings"]) else None for family, (s, k) in medoids.items()}} for cell in grid]
    del system; torch.cuda.empty_cache() if device == "cuda" else None
    discovery = np.flatnonzero((years == reference) & (role == "sae_discovery"))[:max_fit_rows]
    fitted = []
    for seed in seeds[:1]:
        unit = state["cache"].load(f"S/{seed}/{reference}")
        standardized = ((state["predictor"]["embeddings"][discovery] - unit["scaler_mean"]) / unit["scaler_scale"]).astype(np.float32)
        rule = np.flatnonzero((years == reference) & (role == "rule_discovery"))
        for recipe in (mechanism_recipe(state["config"]["sae_seeds"]) if fits is None else fits):
            result = train_diagnostic_sae(standardized, population["patient_id"][rows][discovery], state["config"]["sae"], recipe, device, epochs=epochs)
            with torch.no_grad():
                standardized_rule = torch.as_tensor(((state["predictor"]["embeddings"][rule] - unit["scaler_mean"]) / unit["scaler_scale"]).astype(np.float32))
                new_codes = result["model"].encode(standardized_rule).numpy()
            fitted.append({"recipe": recipe, "loss_first_100_last": [result["losses"][0], result["losses"][min(99, len(result["losses"]) - 1)], result["losses"][-1]],
                "initial_to_trained_cosine": result["initial_to_trained_cosine"], "seconds": result["seconds"], "peak_bytes": result["peak_bytes"],
                "rows": result["rows"], "unique_patients": result["unique_patients"], "active_fraction": float((new_codes > 0).mean()),
                "medoid_recovery": {s: yearly_correspondence(state["codes"][s][rule], new_codes, sorted({k for ss, k in medoids.values() if ss == s}),
                    np.arange(len(rule)), np.ones(len(rule), dtype=bool))["members"] for s in seeds}})
    for d in dictionaries.values():
        d["run"]["model"].to("cpu")
    return {"check": "2 and 5", "posthoc": reference in (2011, 2012), "domain_grid": summary, "later_year": later, "diagnostic_fits": fitted}

def evaluation_suite(population, state, cache, config, reference, device, recipe, development):
    """Frozen final suite for one reference: member transfer, shared-signal tests, both ideas, performance and influence."""
    rule, evaluation, pilot = member_rows(population, state, reference)
    X, names = population["X"][state["rows"]], list(population["feature_names"])
    patients = population["patient_id"][state["rows"]].astype(str)
    features, _ = explanation_features(X, names, **recipe["transform"])
    by_member = {(item["family"], item["seed"], item["factor"]): item for item in development}
    continuous, correspondence = {}, []
    for family_id, frozen in state["references"].items():
        members = [tuple(map(int, member)) for member in state["families"][family_id]["members"]]
        fitted = [by_member.get((family_id, s, k)) for s, k in members]
        chosen = [item["results"][0]["policies"][0] if item and item["results"] else None for item in fitted]
        own = [item if isinstance(item, dict) and item.get("selection_reason") == "selected" else None for item in chosen]
        medoid = members.index(tuple(map(int, frozen["medoid"])))
        fit, tune, _ = inner_split(patients, rule, state, config, reference, family_id, recipe["allocation"])
        if fitted[medoid] and "models" in fitted[medoid].get("continuous", {}):
            continuous[family_id] = {"models": fitted[medoid]["continuous"], "transform": recipe["transform"]}
        values = np.column_stack([state["codes"][s][:, k] for s, k in members])
        cutoffs = [fitted[i]["cutoff"] if fitted[i] else None for i in range(len(members))]
        record = {"family": family_id, "members": members}
        if len(members) == 3 and all(own):
            record["transfer"] = member_transfer([item["rule"] for item in own], features[evaluation], values[evaluation], cutoffs, patients[evaluation])
            common = common_member_rule([item["rule"] for item in own], features[tune], values[tune] > np.asarray(cutoffs, dtype=float))
            record["common_rule"] = {key: value for key, value in common.items() if key != "rule"}
            if common.get("rule") is not None:
                record["common_rule"]["evaluation"] = [fidelity(values[evaluation, j] > cutoffs[j], common["rule"].mask(features[evaluation])) for j in range(3)]
        else:
            record["transfer"] = {"status": "full 3x3 unavailable: needs three members with feasible own rules", "own_feasible": [item is not None for item in own]}
        others = [i for i in range(len(members)) if i != medoid]
        record["shared_signal"] = []
        for i in others:
            (s0, k0), (s1, k1) = members[medoid], members[i]
            g0, g1 = state["codes"][s0][:, [k0]], state["codes"][s1][:, [k1]]
            c0, info0 = matched_nonfamily(g0, state["codes"][s1], [k for s, k in members if s == s1], fit)
            c1, info1 = matched_nonfamily(g1, state["codes"][s0], [k for s, k in members if s == s0], fit)
            mapping = stage(f"mapping/{reference}/{family_id}/{i}", group_mapping, [g0, g1], [c1, c0], patients, fit, tune, evaluation, patients[rule])
            u0, u1 = cache.load(f"S/{s0}/{reference}"), cache.load(f"S/{s1}/{reference}")
            left = group_contribution(g0[evaluation], u0["encoder_weight"][[k0]], u0["scaler_scale"])
            right = group_contribution(g1[evaluation], u1["encoder_weight"][[k1]], u1["scaler_scale"])
            control = group_contribution(c1[evaluation], u0["encoder_weight"][info1["factors"]], u0["scaler_scale"])
            geometry = local_geometry(u0["encoder_weight"][[k0]] * u0["scaler_scale"], u1["encoder_weight"][[k1]] * u1["scaler_scale"],
                                      u0["encoder_weight"][info1["factors"]] * u0["scaler_scale"], u1["encoder_weight"][info0["factors"]] * u1["scaler_scale"])
            record["shared_signal"].append({"member": members[i], "mapping": mapping, "nonfamily": [info0, info1],
                "contribution": contribution_agreement(left, right), "contribution_control": contribution_agreement(left, control), "geometry": geometry})
        correspondence.append(record)
    temporal = stage(f"temporal/{reference}", follow_both_ideas, population, cache, config, reference, device,
                     rule_recipe={"transform": recipe["transform"], "policy": recipe["policy"], "backend": pilot._backend_settings(),
                                  "selection": dict(pilot.rule_selection_config(config["rule"]).__dict__)},
                     continuous_models=continuous)
    if temporal.get("status") != "available":
        return {"reference": reference, "correspondence": correspondence, "temporal": temporal}
    replacements = {}
    for family_id, frozen in state["references"].items():
        members = [tuple(map(int, member)) for member in state["families"][family_id]["members"]]
        seed, factor = map(int, frozen["medoid"])
        other = next(((s, k) for s, k in members if s != seed), None)
        if other is None:
            continue
        unit = cache.load(f"S/{other[0]}/{reference}")
        for year in range(reference, 2016):
            positions = np.flatnonzero((population["year"][state["rows"]] == year) & (state["roles"]["row_role"][state["rows"]] == "evaluation"))
            Xy, values = X[positions], state["codes"][seed][positions, factor]
            for subgroup, mask in (("A", values > frozen["cutoff"]), ("R", frozen["rule"].mask(Xy))):
                replacements[(family_id, year, subgroup)] = group_contribution(state["codes"][other[0]][positions[mask]][:, [other[1]]], unit["encoder_weight"][[other[1]]], unit["scaler_scale"])
    annual = stage(f"performance/{reference}", performance_over_time, population, temporal, cache, reference, device, interventions=True, replacements=replacements,
                   threshold=(recipe["thresholds"].get(str(reference)) or {}).get("threshold"))
    associations = temporal_associations(annual) if isinstance(annual, list) else annual
    return {"reference": reference, "correspondence": correspondence, "temporal": temporal, "annual": annual, "associations": associations}

def load_state(population, cache, config, reference, device):
    state = temporal_inputs(population, cache, config, reference, device)
    if state["status"] == "available":
        state.update(config=config, cache=cache, rules_unit=cache.load(f"D/{reference}/rules"))
    return state

def select_build(caches, configs, reference):
    name = build_for(CFG["variant"], reference)
    return caches[name], configs[name], name

def v1_smoke(population, caches, configs, device, reference=2010, rows=128):
    """Bounded real-data GPU smoke: encoding, head replay/gradients, removal, identity, domain inference, 5-epoch fit, window context."""
    import concept_subgroup_drift_pilot as pilot
    cache, config, _ = select_build(caches, configs, reference)
    # First: TabPFN capacity counts GPU memory that later V1 objects would still hold.
    window = stage("v1/window_context", window_probe, population, config, device, rows=rows)
    state = stage("v1/encode", load_state, population, cache, config, reference, device)
    family_id, frozen = next(iter(state["references"].items()))
    seed, factor = map(int, frozen["medoid"])
    chosen = member_rows(population, state, reference)[1][:rows]
    unit = cache.load(f"S/{seed}/{reference}")
    component = group_contribution(state["codes"][seed][chosen, factor:factor + 1], unit["encoder_weight"][[factor]], unit["scaler_scale"])
    continuation = pilot.load_continuation(cache, reference, device)
    # Smoke reads no outcomes: year 2015 makes the intervention report probability changes only.
    removal = stage("v1/replay_removal", intervention_scores, continuation, state["predictor"]["embeddings"][chosen], state["predictor"]["probabilities"][chosen],
                    state["predictor"]["gradients"][chosen], component, population["patient_id"][state["rows"]][chosen].astype(str), 2015, replicates=4)
    identity = disappearance_identity(state["predictor"]["embeddings"][chosen], unit["encoder_weight"], unit["encoder_bias"], unit["scaler_mean"], unit["scaler_scale"],
        factor, frozen["cutoff"], state["codes"][seed][chosen, factor] > frozen["cutoff"], dict.fromkeys(IDENTITY_FIELDS, "v1"), dict.fromkeys(IDENTITY_FIELDS, "v1"))
    mechanism = stage("v1/domain_fit", mechanism_study, population, state, reference, device, rows_per_period=8,
                      fits=[mechanism_recipe(config["sae_seeds"])[0]], epochs=5, max_fit_rows=rows)
    failed = [name for name, item in STAGES.items() if item["status"] == "failed"]
    if failed:
        raise RuntimeError(f"V1 smoke stages failed: {failed}")
    return {"reference": reference, "family": family_id, "rows": len(chosen), "removal": removal, "identity": identity, "mechanism": mechanism, "window": window}

def develop_mode(population, caches, configs, device):
    sizes = {}
    for r in PRIMARY:
        cache = select_build(caches, configs, r)[0]
        if cache.is_complete(f"D/{r}/rules"):
            sizes[r] = cache.load(f"D/{r}/rules")["rule_rows"]
    choice = mechanism_references(sizes)
    support, development, disappearance, thresholds = {}, {}, {}, {}
    for r in REFERENCES:
        cache, config, _ = select_build(caches, configs, r)
        state = stage(f"state/{r}", load_state, population, cache, config, r, device)
        if state.get("status") != "available":
            development[r] = disappearance[r] = state
            continue
        thresholds[r] = stage(f"threshold/{r}", reference_threshold, population, state, r)
        support[r] = stage(f"support/{r}", reference_support_audit, population, cache, config, r)
        development[r] = reference_development(population, state, config, r)
        disappearance[r] = stage(f"disappearance/{r}", disappearance_reference, population, state, cache, r, device,
                                 mechanism=r in {2011, 2012, choice["reference_support_comparator"]})
        del state
        torch.cuda.empty_cache() if device == "cuda" else None
    recipe = freeze_recipe([item for value in development.values() if isinstance(value, list) for item in value], thresholds)
    atomic_json(OUT / "freeze.json", recipe)
    return {"variant": CFG["variant"], "builds": {r: build_for(CFG["variant"], r) for r in REFERENCES}, "mechanism_references": choice,
            "recipe_hash": recipe["recipe_hash"], "allocation": recipe["allocation"], "thresholds": recipe["thresholds"], "freeze": "freeze.json",
            "exports": [export("support", support), export("development", development), export("disappearance", disappearance)]}

def evaluate_mode(population, caches, configs, device):
    recipe = json.loads(Path(os.environ["PILOT_FOLLOW_UPS_FREEZE"]).read_text())
    if canonical_hash({key: value for key, value in recipe.items() if key != "recipe_hash"}) != recipe["recipe_hash"]:
        raise ValueError("Frozen recipe identity mismatch")
    if recipe["variant"] != CFG["variant"]:
        raise ValueError("Frozen recipe belongs to another variant")
    annual, exports, predictor = [], [], {}
    for r in REFERENCES:
        cache, config, build = select_build(caches, configs, r)
        state = stage(f"state/{r}", load_state, population, cache, config, r, device)
        if state.get("status") != "available":
            exports.append(export(f"evaluate-{r}", state))
            continue
        predictor[r] = stage(f"predictor/{r}", predictor_performance, population, state, r, recipe["thresholds"].get(str(r)))
        development = reference_development(population, state, config, r, recipe=recipe)
        suite = evaluation_suite(population, state, cache, config, r, device, recipe, development)
        suite.update(build=build, block="primary" if r in PRIMARY else "added follow-up 2014/2015")
        exports.append(export(f"evaluate-{r}", suite))
        annual += suite["annual"] if isinstance(suite.get("annual"), list) else []
        del state, suite
        torch.cuda.empty_cache() if device == "cuda" else None
    prospective = secondary_prospective(annual, CFG["prospective_signal"])
    return {"variant": CFG["variant"], "recipe_hash": recipe["recipe_hash"], "prospective_status": prospective["status"],
            "exports": exports + [export("predictor", predictor), export("prospective", prospective)]}


In [ ]:
started = time.monotonic()
now = lambda: datetime.now(timezone.utc).isoformat()
packages = {}
for name in ("torch", "numpy", "pandas", "scikit-learn", "tabpfn", "nbconvert"):
    try:
        packages[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        packages[name] = None
source_names = ("pilot_follow_ups.ipynb", "concept_subgroup_drift_pilot.py",
                "concept_subgroup_drift_pilot.ipynb", "sae.py", "stable_rule_backend.py",
                "semantic_rules.py", "temporal_splits.py")
sources = [file_identity(ROOT / name) for name in source_names if (ROOT / name).is_file()]
launcher_path = Path(os.environ["RPI_RUN_DIR"]) / "run.json" if os.environ.get("RPI_RUN_DIR") else None
manifest = {
    "schema_version": "1", "run_id": OUT.name, "work_item": "pilot-ideas-followups",
    "status": "running", "smoke": MODE in {"v0", "v1"}, "command": sys.argv,
    "host": socket.gethostname(), "started_utc": now(), "ended_utc": None, "seconds": None,
    "git": {"sha": git_read("rev-parse", "HEAD"),
            "branch": git_branch(),
            "dirty": bool(git_read("status", "--porcelain")),
            "dirty_files": git_read("status", "--porcelain").splitlines()},
    "config": {"path": "notebook CFG + environment", "hash": canonical_hash(CFG), "resolved": CFG},
    "data": [], "data_status": "not accessed: synthetic v0" if MODE == "v0" else "pending",
    "sources": sources, "seed": CFG["seed"], "deterministic": CFG["deterministic"],
    "accelerator": {"requested_device": CFG["device"], "resolved_device": None},
    "environment": {"python": sys.version, "executable": sys.executable,
                    "platform": platform.platform(), "packages": packages},
    "outputs": {"metrics": "metrics.json", "stages": "stages.json", "artifacts": []},
    "launcher": json.loads(launcher_path.read_text()) if launcher_path and launcher_path.is_file() else None,
    "error": None,
}
atomic_json(OUT / "run-manifest.json", manifest)
try:
    device = resolve_device(CFG["device"])
    set_seed(CFG["seed"], CFG["deterministic"])
    manifest["accelerator"].update({
        "resolved_device": device, "cuda_available": torch.cuda.is_available(),
        "torch_version": torch.__version__, "cuda_version": torch.version.cuda,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "amp_dtype": CFG["amp"], "tf32_matmul": CFG["tf32"],
        "peak_memory_allocated_bytes": 0, "peak_memory_reserved_bytes": 0,
    })
    if MODE == "v0":
        metrics = focused_setup_checks()
        metrics["patient_support"] = focused_role_checks()
        metrics["reference_explanations"] = focused_reference_checks()
        metrics["correspondence"] = focused_correspondence_checks()
        metrics["disappearance"] = focused_disappearance_checks()
        metrics["temporal_ideas"] = focused_temporal_checks()
        metrics["performance"] = focused_performance_checks()
        metrics["prospective"] = focused_prospective_checks()
    elif MODE == "report":
        manifest["data_status"] = "not accessed: report renders saved develop/evaluate exports"
        metrics = {"report": REPORT_INDEX, "recipes": {variant: run["freeze"]["recipe_hash"] for variant, run in RUNS.items()}}
    else:
        population, caches, configs, provenance = open_inputs()
        manifest["data"] = [{"population_sha256": POPULATION_HASH, "r1_config_hash": R1_CONFIG_HASH, "r1_cache_source_hash": R1_CACHE_SOURCE_HASH,
                             "population_identity": population["fingerprint"]["population_identity"], **provenance}]
        manifest["data_status"] = f"population and read-only {CFG['variant']} unit caches verified"
        metrics = {"v1": v1_smoke, "develop": develop_mode, "evaluate": evaluate_mode}[MODE](population, caches, configs, device)
        peaks = [item.get("peak_reserved", 0) for item in STAGES.values()]
        manifest["accelerator"]["peak_memory_reserved_bytes"] = max(peaks, default=0)
        manifest["accelerator"]["peak_memory_allocated_bytes"] = max((item.get("peak_allocated", 0) for item in STAGES.values()), default=0)
    metrics = compact(metrics)
    atomic_json(OUT / "metrics.json", metrics)
    manifest["status"] = "completed"
    print(json.dumps(metrics, sort_keys=True))
except Exception as error:
    manifest["status"] = "failed"
    manifest["error"] = {"type": type(error).__name__, "message": str(error)}
    raise
finally:
    manifest["ended_utc"] = now()
    manifest["seconds"] = time.monotonic() - started
    STAGES["run"] = {"seconds": manifest["seconds"], "status": manifest["status"]}
    atomic_json(OUT / "stages.json", STAGES)
    atomic_json(OUT / "run-manifest.json", manifest)
